<a href="https://colab.research.google.com/github/veronicalaganaro/Pipeline-som-tico-com-GATK4-Mutect2/blob/main/Somaticos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# 5. Executar o Mutect2 Somático restringindo ao cromossomo 9
!./gatk-4.2.2.0/gatk Mutect2 \
    -R chr9.fa \
    -I tumor_JAK2.bam \
    -I normal_JAK2.bam \
    -normal WP044 \
    --germline-resource af-only-gnomad-chr9.vcf.gz \
    -L 9 \
    -O tumor_JAK2.somatic.vcf.gz

# 6. Calcular a contaminação para estimar artefatos restringindo ao cromossomo 9
!./gatk-4.2.2.0/gatk GetPileupSummaries \
    -I tumor_JAK2.bam \
    -V af-only-gnomad-chr9.vcf.gz \
    -L 9 \
    -O tumor_JAK2.table

!./gatk-4.2.2.0/gatk GetPileupSummaries \
    -I normal_JAK2.bam \
    -V af-only-gnomad-chr9.vcf.gz \
    -L 9 \
    -O normal_JAK2.table

!./gatk-4.2.2.0/gatk CalculateContamination \
    -I tumor_JAK2.table \
    -matched normal_JAK2.table \
    -O tumor_JAK2.contamination.table

# 7. Filtrar as variantes somáticas chamadas pelo Mutect2
!./gatk-4.2.2.0/gatk FilterMutectCalls \
    -R chr9.fa \
    -V tumor_JAK2.somatic.vcf.gz \
    --contamination-table tumor_JAK2.contamination.table \
    -O tumor_JAK2.filtered.vcf.gz

# Exibir as variantes identificadas que passaram nos filtros básicos
!zcat tumor_JAK2.filtered.vcf.gz | grep -v '##' | head -n 30

Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar Mutect2 -R chr9.fa -I tumor_JAK2.bam -I normal_JAK2.bam -normal WP044 --germline-resource af-only-gnomad-chr9.vcf.gz -L 9 -O tumor_JAK2.somatic.vcf.gz
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
02:14:04.740 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
02:14:04.998

In [9]:
!samtools view -h -b /Volumes/Seagate\ Expansion\ Drive/data-lpfap10/projects/proadi/exome/bam/WP043/WP043.bam 9:5021937-5126899 | samtools bam2fq -1 tumor_R1.fq -2 tumor_R2.fq -

[E::hts_open_format] Failed to open file "/Volumes/Seagate Expansion Drive/data-lpfap10/projects/proadi/exome/bam/WP043/WP043.bam" : No such file or directory
samtools view: failed to open "/Volumes/Seagate Expansion Drive/data-lpfap10/projects/proadi/exome/bam/WP043/WP043.bam" for reading: No such file or directory
Failed to read header for "-"


### Método por Código: Fazer Upload do Arquivo BAM e BAI do seu computador
Execute a célula abaixo para selecionar e enviar os arquivos `WP043.bam` e o respectivo índice `.bai`.

In [10]:
from google.colab import files

print("Selecione o arquivo WP043.bam (e o arquivo .bai se necessário):")
uploaded = files.upload()

Selecione o arquivo WP043.bam (e o arquivo .bai se necessário):


Saving normal_JAK2.bam to normal_JAK2 (1).bam


### Executar o Samtools com os arquivos carregados
Após o upload terminar, o arquivo estará disponível na pasta `/content/`. Execute o comando abaixo para realizar a extração das leituras da região de interesse:

In [12]:
!samtools view -h -b /content/somatico/tumor_JAK2.bam 9:5021937-5126899 | samtools bam2fq -1 tumor_R1.fq -2 tumor_R2.fq -

@HWI-ST1374:105:H0NF0ADXX:1:2214:14248:24146_1:N:0:TAAGGCG
AGACACTGCGCCCAGCCCATTTGCAACTTTATTGTTTTCTCTTACAGGCAAATGTTCTGAAAAAGAC
+
:=4<;7<@<956<;;+/==5:<<+8<==;;;.9:==<;=>?>=9==8)>>=)<><=@?8,=;=<?=?
@HWI-ST1374:105:H0NF0ADXX:1:2212:11911:46910_1:N:0:TAAGGCG
GGTCAGATAATCTGCCTCAGATTTCCCAAGGGAATGGTAAAGATACACCTGAAGAACTGGATCTATTTGCTTCAT
+
DF>>=@<<<=<???@???>@=<==???>>@??=><??=<>>@=<<???@?@>>@=>>???=<??<<==>?<9840
@HWI-ST1374:105:H0NF0ADXX:1:2106:4235:45985_1:N:0:TAAGGCG
TTTTAATACTTACCACAAGCTTTAGAAGCAGCAATACAGATTTCTTCTGCAACATACTCCCCAGATGGAAA
+
DD>=;=<<>?=<>?>>>>@@?==<@=>@@>@@>><<>>@=<==??>@@?@>?>><<>?????>@=<??==;
@HWI-ST1374:105:H0NF0ADXX:1:2215:6743:2910_2:N:0:TAAGGCG
GTATTAGCATAGCATAATTAAAACAATCCATGCATAAAAAGGAAAATGCTGTTTTTTAATACTTACCACAAGCTTTAGAAGCAGCAATACAGATTTCTTNN
+
EC<<=;@@<;;@@=;:<==<>==>=>=@7=;@A<===>>=>@==>==?@>@=><>>>====?@=<>@=><;79100..++-/-00..-,0/31/005659:
@HWI-ST1374:105:H0NF0ADXX:1:1103:19036:61762_2:N:0:TAAGGCG
CAACAGAGCAAGACTCAGTCTCAAAAAAAA
+
6C=5;=;>><<><<=><=;=;>;<<<<<<<


In [15]:
!samtools view -h -b /content/somatico/normal_JAK2.bam 9:5021937-5126899 | samtools bam2fq -1 normal_R1.fq -2 normal_R2.fq -

@HWI-ST1374:105:H0NF0ADXX:2:1216:4559:71246_1:N:0:CGTACTA
GATTACAGGTGTGAGACACTGCGCCCAGCCCATTTGTAACTTTATTGTTTTCTCTTACAGGCAAATGTTCT
+
ED?>>@?AA>A<A=@=>>?@@@<@???A@???=>>@==>?@>>==>@=>>>@@@@>=??A?A?>>=B=>@=
@HWI-ST1374:105:H0NF0ADXX:2:1111:14711:95541_1:N:0:CGTACTA
GGAATGGCCTGCCTTACGATGACAGAAATGGAGGGAACATCCACCTCTTCTATATATCAGAATGGTGATATT
+
BB>;;@@@>??>@?=;>;=<@=>>@=>=<@?=@@3=>@==A@??@@@@9??=9<=;<??>>,<@:=A><;;5
@HWI-ST1374:105:H0NF0ADXX:2:1211:14324:30815_1:N:0:CGTACTA
CACCTCTTCTATATATCAGAATGGTGATATTTCTGGAAATGCCAATTCTATGAAGCAAATAGATCCAGTT
+
ED<=>002?><<<<<=>>?=<994<@=:=:::@?-5==?=????===A0<:?;;<@>=,5<6>=?@?@==
@HWI-ST1374:105:H0NF0ADXX:1:1116:3453:9291_1:N:0:CGTACTA
GGTAAAGATACACCTGAAGAACTGGATCTATTTGCTTCATAGAATTGGCATTTCCAGAAATATCACCATTCTG
+
E;><<=@===?>?>>@=>@==?@@?=<A?==>>@A>=A?=>A===>A@A?===A@=A>?>===B?@A?>==;8
@HWI-ST1374:105:H0NF0ADXX:2:1115:7649:41896_1:N:0:CATACTA
ATTCTATGAAGCAAATAGATCCAGTTCTTCAGGTGTATCTTTACCATTCCCTTGGGAAATCTGA
+
BA;<;<7?=>@@=6>9;6<=?=>@>>@?>?>@?38<<==@>:;;=/<>9?@96<

In [19]:
!echo "=== Verificando tumor_JAK2.bam ==="
!samtools quickcheck -v /content/somatico/tumor_JAK2.bam && echo "BAM OK" || echo "BAM Corrompido/Vazio"
!samtools view -H /content/somatico/tumor_JAK2.bam | head -n 5

!echo "\n=== Verificando normal_JAK2.bam ==="
!samtools quickcheck -v /content/somatico/normal_JAK2.bam && echo "BAM OK" || echo "BAM Corrompido/Vazio"
!samtools view -H /content/somatico/normal_JAK2.bam | head -n 5

=== Verificando tumor_JAK2.bam ===
/content/somatico/tumor_JAK2.bam was not identified as sequence data.
/content/somatico/tumor_JAK2.bam
BAM Corrompido/Vazio
[main_samview] fail to read the header from "/content/somatico/tumor_JAK2.bam".
\n=== Verificando normal_JAK2.bam ===
BAM OK
@HD	VN:1.4	GO:none	SO:coordinate
@SQ	SN:1	LN:249250621
@SQ	SN:2	LN:243199373
@SQ	SN:3	LN:198022430
@SQ	SN:4	LN:191154276


In [24]:
!samtools index tumor_JAK2.bam

In [21]:
!git checkout -- /content/somatico/tumor_JAK2.bam /content/somatico/tumor_JAK2.bam.bai
!echo "=== Validando integridade após restauração ==="
!samtools quickcheck -v /content/somatico/tumor_JAK2.bam && echo "BAM restaurado com sucesso!" || echo "Erro ao restaurar BAM."

=== Validando integridade após restauração ===
BAM restaurado com sucesso!


In [26]:
!samtools index normal_JAK2.bam

In [28]:
!zgrep -w "\#" af-only-gnomad.raw.sites.chr.vcf.gz > header

gzip: af-only-gnomad.raw.sites.chr.vcf.gz: No such file or directory


In [29]:
# Extrair o cabeçalho usando o arquivo correto do cromossomo 9 presente na pasta somatico
!zgrep -w "#" /content/somatico/af-only-gnomad-chr9.vcf.gz > /content/somatico/header
!head -n 20 /content/somatico/header

##fileformat=VCFv4.2
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##FORMAT=<ID=AD,Number=R,Type=Integer,Description="Allelic depths for the ref and alt alleles in the order listed">
##FORMAT=<ID=DP,Number=1,Type=Integer,Description="Read Depth">
##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype Quality">
##FORMAT=<ID=PL,Number=G,Type=Integer,Description="Normalized, Phred-scaled likelihoods for genotypes as defined in the VCF specification">
##FILTER=<ID=PASS,Description="">
##FILTER=<ID=PASS,Description="">
##FILTER=<ID=LowQual,Description="">
##FILTER=<ID=InbreedingCoeff,Description="">
##FILTER=<ID=PASS+,Description="">
##FILTER=<ID=PASS+,Description="">
##FILTER=<ID=PASS,Description="">
##FILTER=<ID=PASS,Description="">
##FILTER=<ID=PASS,Description="">
##FILTER=<ID=PASS,Description="">
##INFO=<ID=AC,Number=A,Type=Integer,Description="Allele count in genotypes, for each ALT allele, in the same order as listed">
##INFO=<ID=AF,Number=A,Type=Float,Description

In [31]:
# Extrair a região do gene JAK2 utilizando o recurso correto do gnomAD para o chr9
!zgrep -w "^9" /content/somatico/af-only-gnomad-chr9.vcf.gz | awk '$2>=5021937 && $2<=5126899' > /content/somatico/JAK2.region
# Exibir as primeiras linhas da região extraída
!head -n 10 /content/somatico/JAK2.region

9	5021940	.	A	G	1902.40	PASS	AC=2;AF=4.923E-5
9	5021993	.	A	C	506.46	PASS	AC=1;AF=2.462E-5
9	5022035	.	C	T	1311.40	PASS	AC=2;AF=4.923E-5
9	5022042	rs150159583	A	G	219.46	PASS	AC=1;AF=2.462E-5
9	5022085	.	A	C	572.40	PASS	AC=2;AF=4.923E-5
9	5022091	.	T	C	1962.04	PASS	AC=4;AF=9.846E-5
9	5022098	.	A	G	621.40	PASS	AC=2;AF=4.923E-5
9	5022124	rs138655335	C	A	3052.45	PASS	AC=9;AF=2.215E-4
9	5022125	.	C	T	243.46	PASS	AC=1;AF=2.462E-5
9	5022130	rs143227399	G	A	2152.04	PASS	AC=4;AF=9.846E-5


In [32]:
!cat header JAK2.region > af-only-gnomad-chr9.vcf

In [34]:
# Mover o arquivo para a pasta correta caso tenha sido criado fora, compactar e indexar
!mv af-only-gnomad-chr9.vcf /content/somatico/af-only-gnomad-chr9.vcf 2>/dev/null || true
!bgzip -f /content/somatico/af-only-gnomad-chr9.vcf
!tabix -f -p vcf /content/somatico/af-only-gnomad-chr9.vcf.gz

# Confirmar a criação do arquivo compactado e seu respectivo índice
!ls -lh /content/somatico/af-only-gnomad-chr9.vcf.gz*

/bin/bash: line 1: bgzip: command not found
/bin/bash: line 1: tabix: command not found
-rw-r--r-- 1 root root 138K Nov 25 02:10 /content/somatico/af-only-gnomad-chr9.vcf.gz
-rw-r--r-- 1 root root  184 Nov 25 02:10 /content/somatico/af-only-gnomad-chr9.vcf.gz.tbi


In [35]:
!tabix -p vcf af-only-gnomad-chr9.vcf.gz

/bin/bash: line 1: tabix: command not found


In [36]:
# Indexar o arquivo VCF utilizando o GATK IndexFeatureFile
!./gatk-4.2.2.0/gatk IndexFeatureFile \
    -I /content/somatico/af-only-gnomad-chr9.vcf.gz

# Confirmar a existência do índice gerado (.tbi)
!ls -lh /content/somatico/af-only-gnomad-chr9.vcf.gz*

Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar IndexFeatureFile -I /content/somatico/af-only-gnomad-chr9.vcf.gz
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
02:42:12.651 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
02:42:12.957 INFO  IndexFeatureFile - ------------------------------------------------------------

In [38]:
!sh run.chr13-chr19.sh tumor_JAK2.bam normal_JAK2.bam WP043 WP044
!sh run.chr13-chr19.sh tumor_wp017.bam tumor_wp018.bam WP017 WP018
!sh run.chr13-chr19.sh tumor_wp190.bam tumor_wp191.bam WP190 WP191

Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar Mutect2 -R hg19.fa -I tumor_JAK2.bam -I normal_JAK2.bam -normal WP044 --germline-resource af-only-gnomad-chr13-chr19.vcf.gz -O WP043.somatic.vcf.gz -L hg19.interval_list
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
02:43:02.423 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compress

In [39]:
# 1. Baixar o genoma de referência hg19 completo compactado
!wget -c http://hgdownload.cse.ucsc.edu/goldenPath/hg19/bigZips/hg19.fa.gz

# 2. Descompactar o arquivo no diretório /content/somatico/
!zcat hg19.fa.gz > /content/somatico/hg19.fa

# 3. Criar os índices necessários para o GATK e Samtools
!samtools faidx /content/somatico/hg19.fa
!./gatk-4.2.2.0/gatk CreateSequenceDictionary -R /content/somatico/hg19.fa

# 4. Confirmar a criação dos arquivos de referência
!ls -lh /content/somatico/hg19.*

--2026-11-25 02:45:46--  http://hgdownload.cse.ucsc.edu/goldenPath/hg19/bigZips/hg19.fa.gz
Resolving hgdownload.cse.ucsc.edu (hgdownload.cse.ucsc.edu)... 128.114.119.163
Connecting to hgdownload.cse.ucsc.edu (hgdownload.cse.ucsc.edu)|128.114.119.163|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 948731419 (905M) [application/x-gzip]
Saving to: ‘hg19.fa.gz’

hg19.fa.gz          100%[===================>] 904.78M  17.9MB/s    in 57s     

2026-11-25 02:46:44 (15.8 MB/s) - ‘hg19.fa.gz’ saved [948731419/948731419]

Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar CreateSequenceDictionary -R /content/somatico/hg19.fa
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seem

In [40]:

# variaveis fixas
gnomad="af-only-gnomad-chr13-chr19.vcf.gz"
interval="hg19.interval_list"
genome="hg19.fa"
pon="Mutect2-exome-panel.vcf"

# bam e ids
tumor=$1
id_tumor=$3

# rodando mutect2
./gatk-4.2.2.0/gatk Mutect2 \
	-R $genome \
	-I $tumor \
	--germline-resource $gnomad \
	--panel-of-normals $pon \
	-O $id_tumor.somatic.pon.vcf.gz \
	-L $interval

# getPileup tumor
./gatk-4.2.2.0/gatk GetPileupSummaries \
	-I $tumor \
	-V $gnomad \
	-L $interval \
	-O $id_tumor.pon.table

# CalculateContamination
./gatk-4.2.2.0/gatk CalculateContamination \
	-I $id_tumor.pon.table \
	-O $id_tumor.contamination.pon.table

# FilterMutectCalls
./gatk-4.2.2.0/gatk FilterMutectCalls \
	-R $genome \
	-V $id_tumor.somatic.pon.vcf.gz \
	--contamination-table $id_tumor.contamination.pon.table \
	-O $id_tumor.filtered.pon.vcf.gz

SyntaxError: invalid syntax (1508782459.py, line 8)

In [41]:
# Mudar para o diretório de trabalho correto e executar o pipeline em lote para as três duplas de amostras
%cd /content/somatico/

print("=== Iniciando lote 1/3 (JAK2) ===")
!sh run.chr13-chr19.pon.sh tumor_JAK2.bam normal_JAK2.bam WP043 WP044

print("\n=== Iniciando lote 2/3 (WP017/WP018) ===")
!sh run.chr13-chr19.pon.sh tumor_wp017.bam tumor_wp018.bam WP017 WP018

print("\n=== Iniciando lote 3/3 (WP190/WP191) ===")
!sh run.chr13-chr19.pon.sh tumor_wp190.bam tumor_wp191.bam WP190 WP191

/content/somatico
=== Iniciando lote 1/3 (JAK2) ===
Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar Mutect2 -R hg19.fa -I tumor_JAK2.bam --germline-resource af-only-gnomad-chr13-chr19.vcf.gz --panel-of-normals Mutect2-exome-panel.vcf -O WP043.somatic.pon.vcf.gz -L hg19.interval_list
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
02:48:57.978 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/ga

In [46]:
# Verificar cromossomos do BAM tumor
print("=== BAM WP190 ===")
!samtools view -H /content/somatico/tumor_wp190.bam | grep '^@SQ'

# Verificar cromossomos do BAM normal
print("\n=== BAM WP191 ===")
!samtools view -H /content/somatico/tumor_wp191.bam | grep '^@SQ'

# Verificar referência
print("\n=== REFERÊNCIA HG19 ===")
!cat /content/somatico/hg19.fa.fai

# Verificar intervalos
print("\n=== INTERVAL LIST ===")
!head -20 /content/somatico/hg19.interval_list

=== BAM WP190 ===
@SQ	SN:1	LN:249250621
@SQ	SN:2	LN:243199373
@SQ	SN:3	LN:198022430
@SQ	SN:4	LN:191154276
@SQ	SN:5	LN:180915260
@SQ	SN:6	LN:171115067
@SQ	SN:7	LN:159138663
@SQ	SN:8	LN:146364022
@SQ	SN:9	LN:141213431
@SQ	SN:10	LN:135534747
@SQ	SN:11	LN:135006516
@SQ	SN:12	LN:133851895
@SQ	SN:13	LN:115169878
@SQ	SN:14	LN:107349540
@SQ	SN:15	LN:102531392
@SQ	SN:16	LN:90354753
@SQ	SN:17	LN:81195210
@SQ	SN:18	LN:78077248
@SQ	SN:19	LN:59128983
@SQ	SN:20	LN:63025520
@SQ	SN:21	LN:48129895
@SQ	SN:22	LN:51304566
@SQ	SN:X	LN:155270560
@SQ	SN:Y	LN:59373566
@SQ	SN:MT	LN:16569
@SQ	SN:GL000207.1	LN:4262
@SQ	SN:GL000226.1	LN:15008
@SQ	SN:GL000229.1	LN:19913
@SQ	SN:GL000231.1	LN:27386
@SQ	SN:GL000210.1	LN:27682
@SQ	SN:GL000239.1	LN:33824
@SQ	SN:GL000235.1	LN:34474
@SQ	SN:GL000201.1	LN:36148
@SQ	SN:GL000247.1	LN:36422
@SQ	SN:GL000245.1	LN:36651
@SQ	SN:GL000197.1	LN:37175
@SQ	SN:GL000203.1	LN:37498
@SQ	SN:GL000246.1	LN:38154
@SQ	SN:GL000249.1	LN:38502
@SQ	SN:GL000196.1	LN:38914
@SQ	SN:GL000248.1	LN:39786

In [47]:
!find /content -name "hg19.interval_list" 2>/dev/null

In [48]:
!find /content/somatico -type f \( -name "*.interval_list" -o -name "*.intervals" -o -name "*.list" \)

In [49]:
!ls -lah /content/somatico/

total 4.5G
drwxr-xr-x 4 root root 4.0K Nov 25 02:47  .
drwxr-xr-x 1 root root 4.0K Nov 25 02:10  ..
-rw-r--r-- 1 root root 124K Nov 25 02:10  af-only-gnomad-chr13-chr19.vcf.gz
-rw-r--r-- 1 root root  255 Nov 25 02:10  af-only-gnomad-chr13-chr19.vcf.gz.tbi
-rw-r--r-- 1 root root 591K Nov 25 02:39  af-only-gnomad-chr9.vcf
-rw-r--r-- 1 root root 138K Nov 25 02:10  af-only-gnomad-chr9.vcf.gz
-rw-r--r-- 1 root root  194 Nov 25 02:42  af-only-gnomad-chr9.vcf.gz.tbi
-rw-r--r-- 1 root root  103 Nov 25 02:12  chr9.dict
-rw-r--r-- 1 root root 138M Nov 25 02:11  chr9.fa
-rw-r--r-- 1 root root   20 Nov 25 02:11  chr9.fa.fai
-rw-r--r-- 1 root root  38M Mar 20  2009  chr9.fa.gz
drwxr-xr-x 4 root root 4.0K Aug 18  2021  gatk-4.2.2.0
-rw-r--r-- 1 root root 454M Dec  7  2021  gatk-4.2.2.0.zip
drwxr-xr-x 8 root root 4.0K Nov 25 02:34  .git
-rw-r--r-- 1 root root 3.9K Nov 25 02:37  header
-rw-r--r-- 1 root root 9.3K Nov 25 02:48  hg19.dict
-rw-r--r-- 1 root root 3.0G Nov 25 02:47  hg19.fa
-rw-r--r-- 1 ro

In [50]:
print("=== CONTIGS DA REFERÊNCIA ===")
!cut -f1,2 /content/somatico/hg19.fa.fai

print("\n=== CONTIGS DO BAM WP190 ===")
!samtools view -H /content/somatico/tumor_wp190.bam | grep '^@SQ'

print("\n=== ARQUIVOS PON ===")
!ls -lh /content/somatico/*Pon* /content/somatico/*panel* 2>/dev/null

=== CONTIGS DA REFERÊNCIA ===
chr1	249250621
chr2	243199373
chr3	198022430
chr4	191154276
chr5	180915260
chr6	171115067
chr7	159138663
chrX	155270560
chr8	146364022
chr9	141213431
chr10	135534747
chr11	135006516
chr12	133851895
chr13	115169878
chr14	107349540
chr15	102531392
chr16	90354753
chr17	81195210
chr18	78077248
chr20	63025520
chrY	59373566
chr19	59128983
chr22	51304566
chr21	48129895
chr6_ssto_hap7	4928567
chr6_mcf_hap5	4833398
chr6_cox_hap2	4795371
chr6_mann_hap4	4683263
chr6_apd_hap1	4622290
chr6_qbl_hap6	4611984
chr6_dbb_hap3	4610396
chr17_ctg5_hap1	1680828
chr4_ctg9_hap1	590426
chr1_gl000192_random	547496
chrUn_gl000225	211173
chr4_gl000194_random	191469
chr4_gl000193_random	189789
chr9_gl000200_random	187035
chrUn_gl000222	186861
chrUn_gl000212	186858
chr7_gl000195_random	182896
chrUn_gl000223	180455
chrUn_gl000224	179693
chrUn_gl000219	179198
chr17_gl000205_random	174588
chrUn_gl000215	172545
chrUn_gl000216	172294
chrUn_gl000217	172149
chr9_gl000199_random	169874
chrUn_gl

In [51]:
%cd /content/somatico

!./gatk-4.2.2.0/gatk ScatterIntervalsByNs \
-R hg19.fa \
-O hg19.interval_list \
-OT ACGT

/content/somatico
Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar ScatterIntervalsByNs -R hg19.fa -O hg19.interval_list -OT ACGT
[0.014s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.014s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
03:39:05.000 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
[Wed Nov 25 03:39:05 UTC 2026] ScatterIntervalsByNs --OUTPUT hg19.interval_list --

In [52]:
!ls -lh hg19.interval_list

-rw-r--r-- 1 root root 26K Nov 25 03:41 hg19.interval_list


In [53]:
%cd /content/somatico

!ls -lh *Pon* *pon* *panel* 2>/dev/null

/content/somatico
-rw-r--r-- 1 root root 882 Nov 25 02:10  run.chr13-chr19.pon.sh
-rw-r--r-- 1 root root 14K Nov 25 02:10  somatic.Pon.vcf


In [54]:
!head -10 somatic.Pon.vcf

##fileformat=VCFv4.2
##FORMAT=<ID=AD,Number=R,Type=Integer,Description="Allelic depths for the ref and alt alleles in the order listed">
##FORMAT=<ID=AF,Number=A,Type=Float,Description="Allele fractions of alternate alleles in the tumor">
##FORMAT=<ID=DP,Number=1,Type=Integer,Description="Approximate read depth (reads with MQ=255 or with bad mates are filtered)">
##FORMAT=<ID=F1R2,Number=R,Type=Integer,Description="Count of reads in F1R2 pair orientation supporting each allele">
##FORMAT=<ID=F2R1,Number=R,Type=Integer,Description="Count of reads in F2R1 pair orientation supporting each allele">
##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype Quality">
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##FORMAT=<ID=PGT,Number=1,Type=String,Description="Physical phasing haplotype information, describing how the alternate alleles are phased in relation to one another; will always be heterozygous and is not intended to describe called alleles">
##FORMAT=<ID=PID,Number

In [55]:
%cd /content/somatico

print("=== AMOSTRAS ===")
!grep '^#CHROM' somatic.Pon.vcf

print("\n=== CONTIGS ===")
!grep '^##contig' somatic.Pon.vcf

print("\n=== QUANTIDADE DE VARIANTES ===")
!grep -vc '^#' somatic.Pon.vcf

print("\n=== PRIMEIRAS VARIANTES ===")
!grep -v '^#' somatic.Pon.vcf | head -5

/content/somatico
=== AMOSTRAS ===
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	WP043

=== CONTIGS ===
##contig=<ID=1,length=249250621>
##contig=<ID=2,length=243199373>
##contig=<ID=3,length=198022430>
##contig=<ID=4,length=191154276>
##contig=<ID=5,length=180915260>
##contig=<ID=6,length=171115067>
##contig=<ID=7,length=159138663>
##contig=<ID=8,length=146364022>
##contig=<ID=9,length=141213431>
##contig=<ID=10,length=135534747>
##contig=<ID=11,length=135006516>
##contig=<ID=12,length=133851895>
##contig=<ID=13,length=115169878>
##contig=<ID=14,length=107349540>
##contig=<ID=15,length=102531392>
##contig=<ID=16,length=90354753>
##contig=<ID=17,length=81195210>
##contig=<ID=18,length=78077248>
##contig=<ID=19,length=59128983>
##contig=<ID=20,length=63025520>
##contig=<ID=21,length=48129895>
##contig=<ID=22,length=51304566>
##contig=<ID=X,length=155270560>
##contig=<ID=Y,length=59373566>
##contig=<ID=MT,length=16569>

=== QUANTIDADE DE VARIANTES ===
22

=== PRIMEIRAS VARIANTES ===
9	50

In [56]:
%cd /content/somatico

print("=== REFERÊNCIA ===")
!cut -f1,2 hg19.fa.fai

print("\n=== BAM TUMOR WP190 ===")
!samtools view -H tumor_wp190.bam | grep '^@SQ'

print("\n=== BAM NORMAL WP191 ===")
!samtools view -H tumor_wp191.bam | grep '^@SQ'

/content/somatico
=== REFERÊNCIA ===
chr1	249250621
chr2	243199373
chr3	198022430
chr4	191154276
chr5	180915260
chr6	171115067
chr7	159138663
chrX	155270560
chr8	146364022
chr9	141213431
chr10	135534747
chr11	135006516
chr12	133851895
chr13	115169878
chr14	107349540
chr15	102531392
chr16	90354753
chr17	81195210
chr18	78077248
chr20	63025520
chrY	59373566
chr19	59128983
chr22	51304566
chr21	48129895
chr6_ssto_hap7	4928567
chr6_mcf_hap5	4833398
chr6_cox_hap2	4795371
chr6_mann_hap4	4683263
chr6_apd_hap1	4622290
chr6_qbl_hap6	4611984
chr6_dbb_hap3	4610396
chr17_ctg5_hap1	1680828
chr4_ctg9_hap1	590426
chr1_gl000192_random	547496
chrUn_gl000225	211173
chr4_gl000194_random	191469
chr4_gl000193_random	189789
chr9_gl000200_random	187035
chrUn_gl000222	186861
chrUn_gl000212	186858
chr7_gl000195_random	182896
chrUn_gl000223	180455
chrUn_gl000224	179693
chrUn_gl000219	179198
chr17_gl000205_random	174588
chrUn_gl000215	172545
chrUn_gl000216	172294
chrUn_gl000217	172149
chr9_gl000199_random	169874
c

In [57]:
%cd /content/somatico

!grep '^>' hg19.fa | head -30

/content/somatico
>chr1
>chr2
>chr3
>chr4
>chr5
>chr6
>chr7
>chrX
>chr8
>chr9
>chr10
>chr11
>chr12
>chr13
>chr14
>chr15
>chr16
>chr17
>chr18
>chr20
>chrY
>chr19
>chr22
>chr21
>chr6_ssto_hap7
>chr6_mcf_hap5
>chr6_cox_hap2
>chr6_mann_hap4
>chr6_apd_hap1
>chr6_qbl_hap6


In [60]:
%cd /content/somatico

import gzip
import subprocess

print("=== CONTIGS DO GNOMAD ===")
with gzip.open("af-only-gnomad-chr13-chr19.vcf.gz", "rt") as f:
    for line in f:
        if line.startswith("##contig"):
            print(line.strip())
        if line.startswith("#CHROM"):
            break

print("\n=== PRIMEIRAS VARIANTES DO GNOMAD ===")
with gzip.open("af-only-gnomad-chr13-chr19.vcf.gz", "rt") as f:
    count = 0
    for line in f:
        if not line.startswith("#"):
            print(line.strip())
            count += 1
            if count == 5:
                break

print("\n=== IDENTIFICAÇÃO DAS AMOSTRAS NOS BAMs ===")
for bam in ["tumor_wp190.bam", "tumor_wp191.bam"]:
    print("\n", bam)
    result = subprocess.run(
        ["samtools", "view", "-H", bam],
        capture_output=True, text=True
    )
    for line in result.stdout.splitlines():
        if line.startswith("@RG"):
            print(line)

print("\n=== CONTIGS RESTANTES DA REFERÊNCIA ===")
with open("hg19.fa", "r") as f:
    headers = [line.strip() for line in f if line.startswith(">")]
print("\n".join(headers[-20:]))

/content/somatico
=== CONTIGS DO GNOMAD ===

=== PRIMEIRAS VARIANTES DO GNOMAD ===
13	28578060	.	G	A	399.46	PASS	AC=1;AF=2.462E-5
13	28578072	.	G	A	115.46	PASS	AC=1;AF=2.462E-5
13	28578088	.	T	C	500.46	PASS	AC=1;AF=2.462E-5
13	28578093	.	G	T	1018.39	PASS	AC=2;AF=4.923E-5
13	28578109	.	T	C	3719.37	PASS	AC=10;AF=2.462E-4

=== IDENTIFICAÇÃO DAS AMOSTRAS NOS BAMs ===

 tumor_wp190.bam
@RG	ID:test8	PL:illumina	LB:test_8_WP190	SM:WP190	CN:NGSColumbia

 tumor_wp191.bam
@RG	ID:03615	PL:illumina	LB:0361_5_WP191	SM:WP191	CN:NGSColumbia

=== CONTIGS RESTANTES DA REFERÊNCIA ===
>chr11_gl000202_random
>chrUn_gl000238
>chrUn_gl000244
>chrUn_gl000248
>chr8_gl000196_random
>chrUn_gl000249
>chrUn_gl000246
>chr17_gl000203_random
>chr8_gl000197_random
>chrUn_gl000245
>chrUn_gl000247
>chr9_gl000201_random
>chrUn_gl000235
>chrUn_gl000239
>chr21_gl000210_random
>chrUn_gl000231
>chrUn_gl000229
>chrM
>chrUn_gl000226
>chr18_gl000207_random


In [61]:
%cd /content/somatico

import subprocess
import gzip

print("=== REFERÊNCIA hg19 ===")
for line in open("hg19.fa.fai"):
    campos = line.strip().split("\t")
    if campos[0] in ["chr13", "chr19"]:
        print(campos[0], campos[1])

print("\n=== BAM WP190 ===")
for line in subprocess.check_output(
    ["samtools", "view", "-H", "tumor_wp190.bam"], text=True
).splitlines():
    if line.startswith("@SQ") and any(x in line for x in ["SN:13\t", "SN:19\t"]):
        print(line)

print("\n=== BAM WP191 ===")
for line in subprocess.check_output(
    ["samtools", "view", "-H", "tumor_wp191.bam"], text=True
).splitlines():
    if line.startswith("@SQ") and any(x in line for x in ["SN:13\t", "SN:19\t"]):
        print(line)

print("\n=== CABEÇALHO GNOMAD ===")
with gzip.open("af-only-gnomad-chr13-chr19.vcf.gz", "rt") as f:
    for i, line in enumerate(f):
        if line.startswith("#"):
            print(line.strip())
        if i >= 30:
            break

/content/somatico
=== REFERÊNCIA hg19 ===
chr13 115169878
chr19 59128983

=== BAM WP190 ===
@SQ	SN:13	LN:115169878
@SQ	SN:19	LN:59128983

=== BAM WP191 ===
@SQ	SN:13	LN:115169878
@SQ	SN:19	LN:59128983

=== CABEÇALHO GNOMAD ===
##fileformat=VCFv4.2
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##FORMAT=<ID=AD,Number=R,Type=Integer,Description="Allelic depths for the ref and alt alleles in the order listed">
##FORMAT=<ID=DP,Number=1,Type=Integer,Description="Read Depth">
##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype Quality">
##FORMAT=<ID=PL,Number=G,Type=Integer,Description="Normalized, Phred-scaled likelihoods for genotypes as defined in the VCF specification">
##FILTER=<ID=PASS,Description="">
##FILTER=<ID=PASS,Description="">
##FILTER=<ID=LowQual,Description="">
##FILTER=<ID=InbreedingCoeff,Description="">
##FILTER=<ID=PASS+,Description="">
##FILTER=<ID=PASS+,Description="">
##FILTER=<ID=PASS,Description="">
##FILTER=<ID=PASS,Description="">
##FILTER=<ID=

In [62]:
%cd /content/somatico

import subprocess

print("=== TODOS OS CONTIGS DO BAM WP190 ===")
header = subprocess.check_output(
    ["samtools", "view", "-H", "tumor_wp190.bam"], text=True
)

for line in header.splitlines():
    if line.startswith("@SQ"):
        print(line)

print("\n=== TODOS OS CONTIGS DA REFERÊNCIA ===")
with open("hg19.fa.fai") as f:
    for line in f:
        print(line.strip())

/content/somatico
=== TODOS OS CONTIGS DO BAM WP190 ===
@SQ	SN:1	LN:249250621
@SQ	SN:2	LN:243199373
@SQ	SN:3	LN:198022430
@SQ	SN:4	LN:191154276
@SQ	SN:5	LN:180915260
@SQ	SN:6	LN:171115067
@SQ	SN:7	LN:159138663
@SQ	SN:8	LN:146364022
@SQ	SN:9	LN:141213431
@SQ	SN:10	LN:135534747
@SQ	SN:11	LN:135006516
@SQ	SN:12	LN:133851895
@SQ	SN:13	LN:115169878
@SQ	SN:14	LN:107349540
@SQ	SN:15	LN:102531392
@SQ	SN:16	LN:90354753
@SQ	SN:17	LN:81195210
@SQ	SN:18	LN:78077248
@SQ	SN:19	LN:59128983
@SQ	SN:20	LN:63025520
@SQ	SN:21	LN:48129895
@SQ	SN:22	LN:51304566
@SQ	SN:X	LN:155270560
@SQ	SN:Y	LN:59373566
@SQ	SN:MT	LN:16569
@SQ	SN:GL000207.1	LN:4262
@SQ	SN:GL000226.1	LN:15008
@SQ	SN:GL000229.1	LN:19913
@SQ	SN:GL000231.1	LN:27386
@SQ	SN:GL000210.1	LN:27682
@SQ	SN:GL000239.1	LN:33824
@SQ	SN:GL000235.1	LN:34474
@SQ	SN:GL000201.1	LN:36148
@SQ	SN:GL000247.1	LN:36422
@SQ	SN:GL000245.1	LN:36651
@SQ	SN:GL000197.1	LN:37175
@SQ	SN:GL000203.1	LN:37498
@SQ	SN:GL000246.1	LN:38154
@SQ	SN:GL000249.1	LN:38502
@SQ	SN:GL000196

In [63]:
import subprocess

for bam in ["tumor_wp190.bam", "tumor_wp191.bam"]:
    print("\n###", bam)
    header = subprocess.check_output(
        ["samtools", "view", "-H", bam], text=True
    )
    for line in header.splitlines():
        if line.startswith("@SQ"):
            print(line)


### tumor_wp190.bam
@SQ	SN:1	LN:249250621
@SQ	SN:2	LN:243199373
@SQ	SN:3	LN:198022430
@SQ	SN:4	LN:191154276
@SQ	SN:5	LN:180915260
@SQ	SN:6	LN:171115067
@SQ	SN:7	LN:159138663
@SQ	SN:8	LN:146364022
@SQ	SN:9	LN:141213431
@SQ	SN:10	LN:135534747
@SQ	SN:11	LN:135006516
@SQ	SN:12	LN:133851895
@SQ	SN:13	LN:115169878
@SQ	SN:14	LN:107349540
@SQ	SN:15	LN:102531392
@SQ	SN:16	LN:90354753
@SQ	SN:17	LN:81195210
@SQ	SN:18	LN:78077248
@SQ	SN:19	LN:59128983
@SQ	SN:20	LN:63025520
@SQ	SN:21	LN:48129895
@SQ	SN:22	LN:51304566
@SQ	SN:X	LN:155270560
@SQ	SN:Y	LN:59373566
@SQ	SN:MT	LN:16569
@SQ	SN:GL000207.1	LN:4262
@SQ	SN:GL000226.1	LN:15008
@SQ	SN:GL000229.1	LN:19913
@SQ	SN:GL000231.1	LN:27386
@SQ	SN:GL000210.1	LN:27682
@SQ	SN:GL000239.1	LN:33824
@SQ	SN:GL000235.1	LN:34474
@SQ	SN:GL000201.1	LN:36148
@SQ	SN:GL000247.1	LN:36422
@SQ	SN:GL000245.1	LN:36651
@SQ	SN:GL000197.1	LN:37175
@SQ	SN:GL000203.1	LN:37498
@SQ	SN:GL000246.1	LN:38154
@SQ	SN:GL000249.1	LN:38502
@SQ	SN:GL000196.1	LN:38914
@SQ	SN:GL000248.1	LN:39

In [64]:
%cd /content/somatico

import subprocess

# Ler os contigs da referência
fai = {}
with open("hg19.fa.fai") as f:
    for line in f:
        nome, tamanho = line.split("\t")[:2]
        fai[nome] = int(tamanho)

# Ler os contigs do BAM
header = subprocess.check_output(
    ["samtools", "view", "-H", "tumor_wp190.bam"],
    text=True
)

bam = {}
for line in header.splitlines():
    if line.startswith("@SQ"):
        campos = dict(x.split(":", 1) for x in line.split("\t")[1:])
        bam[campos["SN"]] = int(campos["LN"])

# Fazer o mapeamento dos nomes
def converter(nome):
    if nome.startswith("chrUn_"):
        return nome.replace("chrUn_", "")
    if "_gl" in nome:
        return nome.split("_")[-1]
    if nome == "chrM":
        return "MT"
    if nome.startswith("chr"):
        return nome[3:]
    return nome

print("=== DIFERENÇAS ENCONTRADAS ===")

for nome, tamanho in fai.items():
    convertido = converter(nome)

    if convertido not in bam:
        print("AUSENTE NO BAM:", nome, "->", convertido)
    elif tamanho != bam[convertido]:
        print("DIFERENÇA DE TAMANHO:", nome, "->", convertido,
              "| FASTA:", tamanho, "| BAM:", bam[convertido])

print("\n=== CONTIGS DO BAM SEM CORRESPONDÊNCIA ===")
for nome in bam:
    if nome not in [converter(x) for x in fai]:
        print(nome)

/content/somatico
=== DIFERENÇAS ENCONTRADAS ===
AUSENTE NO BAM: chr6_ssto_hap7 -> 6_ssto_hap7
AUSENTE NO BAM: chr6_mcf_hap5 -> 6_mcf_hap5
AUSENTE NO BAM: chr6_cox_hap2 -> 6_cox_hap2
AUSENTE NO BAM: chr6_mann_hap4 -> 6_mann_hap4
AUSENTE NO BAM: chr6_apd_hap1 -> 6_apd_hap1
AUSENTE NO BAM: chr6_qbl_hap6 -> 6_qbl_hap6
AUSENTE NO BAM: chr6_dbb_hap3 -> 6_dbb_hap3
AUSENTE NO BAM: chr17_ctg5_hap1 -> 17_ctg5_hap1
AUSENTE NO BAM: chr4_ctg9_hap1 -> 4_ctg9_hap1
AUSENTE NO BAM: chr1_gl000192_random -> random
AUSENTE NO BAM: chrUn_gl000225 -> gl000225
AUSENTE NO BAM: chr4_gl000194_random -> random
AUSENTE NO BAM: chr4_gl000193_random -> random
AUSENTE NO BAM: chr9_gl000200_random -> random
AUSENTE NO BAM: chrUn_gl000222 -> gl000222
AUSENTE NO BAM: chrUn_gl000212 -> gl000212
AUSENTE NO BAM: chr7_gl000195_random -> random
AUSENTE NO BAM: chrUn_gl000223 -> gl000223
AUSENTE NO BAM: chrUn_gl000224 -> gl000224
AUSENTE NO BAM: chrUn_gl000219 -> gl000219
AUSENTE NO BAM: chr17_gl000205_random -> random
AUSE

In [65]:
%cd /content/somatico

import re
import subprocess

fai = {}
with open("hg19.fa.fai") as f:
    for line in f:
        nome, tamanho = line.strip().split("\t")[:2]
        fai[nome] = int(tamanho)

header = subprocess.check_output(
    ["samtools", "view", "-H", "tumor_wp190.bam"],
    text=True
)

bam = {}
for line in header.splitlines():
    if line.startswith("@SQ"):
        campos = dict(x.split(":", 1) for x in line.split("\t")[1:])
        bam[campos["SN"]] = int(campos["LN"])

def converter(nome):
    match = re.search(r"gl(\d{6})", nome, re.IGNORECASE)
    if match:
        return "GL" + match.group(1) + ".1"
    if nome == "chrM":
        return "MT"
    if nome.startswith("chr"):
        return nome[3:]
    return nome

print("=== CONTIGS DO BAM SEM CORRESPONDÊNCIA ===")
for nome, tamanho in bam.items():
    correspondencias = [
        (ref, tam) for ref, tam in fai.items()
        if converter(ref) == nome
    ]
    if not correspondencias:
        print("AUSENTE:", nome)
    elif not any(tam == tamanho for ref, tam in correspondencias):
        print("TAMANHO DIFERENTE:", nome, tamanho, correspondencias)

print("\n=== CONTIGS COM CORRESPONDÊNCIA E TAMANHO DIFERENTE ===")
for ref, tamanho in fai.items():
    convertido = converter(ref)
    if convertido in bam and tamanho != bam[convertido]:
        print(ref, "->", convertido,
              "| FASTA:", tamanho, "| BAM:", bam[convertido])

/content/somatico
=== CONTIGS DO BAM SEM CORRESPONDÊNCIA ===
TAMANHO DIFERENTE: MT 16569 [('chrM', 16571)]

=== CONTIGS COM CORRESPONDÊNCIA E TAMANHO DIFERENTE ===
chrM -> MT | FASTA: 16571 | BAM: 16569


In [66]:
%cd /content/somatico

import subprocess
import gzip
import os

# Baixar somente os cromossomos 13 e 19
for cromossomo in ["13", "19"]:
    arquivo = f"chr{cromossomo}.fa.gz"
    url = f"https://hgdownload.soe.ucsc.edu/goldenPath/hg19/chromosomes/{arquivo}"

    if not os.path.exists(arquivo):
        subprocess.run(["wget", "-c", url], check=True)

# Concatenar e retirar o prefixo chr dos cabeçalhos
referencia = "hg19_chr13_chr19.fa"

with open(referencia, "w") as saida:
    for cromossomo in ["13", "19"]:
        with gzip.open(f"chr{cromossomo}.fa.gz", "rt") as entrada:
            for linha in entrada:
                if linha.startswith(">"):
                    linha = linha.replace(">chr", ">", 1)
                saida.write(linha)

print("Referência criada:", referencia)

/content/somatico
Referência criada: hg19_chr13_chr19.fa


In [67]:
%cd /content/somatico

import subprocess

ref = "hg19_chr13_chr19.fa"

subprocess.run(["samtools", "faidx", ref], check=True)

subprocess.run([
    "./gatk-4.2.2.0/gatk",
    "CreateSequenceDictionary",
    "-R", ref,
    "-O", "hg19_chr13_chr19.dict"
], check=True)

subprocess.run([
    "./gatk-4.2.2.0/gatk",
    "ScatterIntervalsByNs",
    "-R", ref,
    "-O", "hg19_chr13_chr19.interval_list",
    "-OT", "ACGT"
], check=True)

print("Índices e intervalos criados.")

/content/somatico
Índices e intervalos criados.


In [68]:
%cd /content/somatico

print("FASTA:")
!grep '^>' hg19_chr13_chr19.fa

print("\nÍNDICE:")
!cat hg19_chr13_chr19.fa.fai

print("\nINTERVALOS:")
!head hg19_chr13_chr19.interval_list

/content/somatico
FASTA:
>13
>19

ÍNDICE:
13	115169878	4	50	51
19	59128983	117473284	50	51

INTERVALOS:
@HD	VN:1.6	SO:coordinate
@SQ	SN:13	LN:115169878	M5:283f8d7892baa81b510a015719ca7b0b	UR:file:/content/somatico/hg19_chr13_chr19.fa
@SQ	SN:19	LN:59128983	M5:1aacd71f30db8e561810913e0b72636d	UR:file:/content/somatico/hg19_chr13_chr19.fa
13	19020001	86760324	+	ACGTmer
13	86910325	112353994	+	ACGTmer
13	112503995	114325993	+	ACGTmer
13	114425994	114639948	+	ACGTmer
13	114739949	115109878	+	ACGTmer
19	60001	7346004	+	ACGTmer
19	7396005	8687198	+	ACGTmer


In [69]:
%cd /content/somatico

print("=== SCRIPT PAREADO ===")
print(open("run.chr13-chr19.sh").read())

print("\n=== SCRIPT COM PANEL OF NORMALS ===")
print(open("run.chr13-chr19.pon.sh").read())

/content/somatico
=== SCRIPT PAREADO ===
#####
# rodando o par wp190 (tumor) wp191 (normal)
#####

# variaveis fixas
gnomad="af-only-gnomad-chr13-chr19.vcf.gz"
interval="hg19.interval_list"
genome="hg19.fa"

# bam e ids 
tumor=$1
normal=$2
id_tumor=$3
id_normal=$4

# rodando mutect2
./gatk-4.2.2.0/gatk Mutect2 \
	-R $genome \
	-I $tumor \
	-I $normal \
	-normal $id_normal \
	--germline-resource $gnomad \
	-O $id_tumor.somatic.vcf.gz \
	-L $interval

# getPileup tumor
./gatk-4.2.2.0/gatk GetPileupSummaries \
	-I $tumor \
	-V $gnomad \
	-L $interval \
	-O $id_tumor.table

# getPileup normal
./gatk-4.2.2.0/gatk GetPileupSummaries \
	-I $normal \
	-V $gnomad \
	-L $interval \
	-O $id_normal.table

# CalculateContamination
./gatk-4.2.2.0/gatk CalculateContamination \
	-I $id_tumor.table \
	-matched $id_normal.table \
	-O $id_tumor.contamination.table

# FilterMutectCalls
./gatk-4.2.2.0/gatk FilterMutectCalls \
	-R $genome \
	-V $id_tumor.somatic.vcf.gz \
	--contamination-table $id_tumor.con

In [70]:
%cd /content/somatico

print(open("run.chr13-chr19.sh").read())

/content/somatico
#####
# rodando o par wp190 (tumor) wp191 (normal)
#####

# variaveis fixas
gnomad="af-only-gnomad-chr13-chr19.vcf.gz"
interval="hg19.interval_list"
genome="hg19.fa"

# bam e ids 
tumor=$1
normal=$2
id_tumor=$3
id_normal=$4

# rodando mutect2
./gatk-4.2.2.0/gatk Mutect2 \
	-R $genome \
	-I $tumor \
	-I $normal \
	-normal $id_normal \
	--germline-resource $gnomad \
	-O $id_tumor.somatic.vcf.gz \
	-L $interval

# getPileup tumor
./gatk-4.2.2.0/gatk GetPileupSummaries \
	-I $tumor \
	-V $gnomad \
	-L $interval \
	-O $id_tumor.table

# getPileup normal
./gatk-4.2.2.0/gatk GetPileupSummaries \
	-I $normal \
	-V $gnomad \
	-L $interval \
	-O $id_normal.table

# CalculateContamination
./gatk-4.2.2.0/gatk CalculateContamination \
	-I $id_tumor.table \
	-matched $id_normal.table \
	-O $id_tumor.contamination.table

# FilterMutectCalls
./gatk-4.2.2.0/gatk FilterMutectCalls \
	-R $genome \
	-V $id_tumor.somatic.vcf.gz \
	--contamination-table $id_tumor.contamination.table \
	-O 

In [71]:
%cd /content/somatico

!./gatk-4.2.2.0/gatk Mutect2 \
    -R hg19_chr13_chr19.fa \
    -I tumor_wp190.bam \
    -I tumor_wp191.bam \
    -normal WP191 \
    --germline-resource af-only-gnomad-chr13-chr19.vcf.gz \
    -O WP190.somatic.vcf.gz \
    -L hg19_chr13_chr19.interval_list

/content/somatico
Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar Mutect2 -R hg19_chr13_chr19.fa -I tumor_wp190.bam -I tumor_wp191.bam -normal WP191 --germline-resource af-only-gnomad-chr13-chr19.vcf.gz -O WP190.somatic.vcf.gz -L hg19_chr13_chr19.interval_list
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
04:05:55.885 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-loca

In [72]:
%cd /content/somatico

import os

print("VCF:", os.path.exists("WP190.somatic.vcf.gz"))
print("Índice:", os.path.exists("WP190.somatic.vcf.gz.tbi"))
print("Stats:", os.path.exists("WP190.somatic.vcf.gz.stats"))

/content/somatico
VCF: True
Índice: True
Stats: True


In [73]:
!./gatk-4.2.2.0/gatk GetPileupSummaries \
    -I tumor_wp190.bam \
    -V af-only-gnomad-chr13-chr19.vcf.gz \
    -L hg19_chr13_chr19.interval_list \
    -O WP190.table

Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar GetPileupSummaries -I tumor_wp190.bam -V af-only-gnomad-chr13-chr19.vcf.gz -L hg19_chr13_chr19.interval_list -O WP190.table
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
04:08:09.709 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
04:08:10.250 INFO  GetPileupSummaries -

In [74]:
!./gatk-4.2.2.0/gatk GetPileupSummaries \
    -I tumor_wp191.bam \
    -V af-only-gnomad-chr13-chr19.vcf.gz \
    -L hg19_chr13_chr19.interval_list \
    -O WP191.table

Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar GetPileupSummaries -I tumor_wp191.bam -V af-only-gnomad-chr13-chr19.vcf.gz -L hg19_chr13_chr19.interval_list -O WP191.table
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
04:08:28.445 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
04:08:28.945 INFO  GetPileupSummaries -

In [75]:
!./gatk-4.2.2.0/gatk GetPileupSummaries \
    -I tumor_wp191.bam \
    -V af-only-gnomad-chr13-chr19.vcf.gz \
    -L hg19_chr13_chr19.interval_list \
    -O WP191.table

Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar GetPileupSummaries -I tumor_wp191.bam -V af-only-gnomad-chr13-chr19.vcf.gz -L hg19_chr13_chr19.interval_list -O WP191.table
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
04:09:21.088 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
04:09:21.408 INFO  GetPileupSummaries -

In [76]:
!./gatk-4.2.2.0/gatk CalculateContamination \
    -I WP190.table \
    -matched WP191.table \
    -O WP190.contamination.table

Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar CalculateContamination -I WP190.table -matched WP191.table -O WP190.contamination.table
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
04:10:05.283 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
04:10:05.561 INFO  CalculateContamination - -------------------------------

In [77]:
with open("WP190.contamination.table") as f:
    print(f.read())

sample	contamination	error
WP190	0.0	4.76837158203125E-7



In [79]:
!./gatk-4.2.2.0/gatk FilterMutectCalls \
    -R hg19_chr13_chr19.fa \
    -V WP190.somatic.vcf.gz \
    --contamination-table WP190.contamination.table \
    -O WP190.filtered.vcf.gz

Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar FilterMutectCalls -R hg19_chr13_chr19.fa -V WP190.somatic.vcf.gz --contamination-table WP190.contamination.table -O WP190.filtered.vcf.gz
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
04:11:57.141 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
04:11:57.424 INFO  Filter

In [80]:
!zcat WP190.filtered.vcf.gz | grep -v "^#" | head -30

13	28588649	.	C	T	.	weak_evidence	AS_FilterStatus=weak_evidence;AS_SB_TABLE=118,163|1,2;DP=298;ECNT=1;GERMQ=93;MBQ=31,34;MFRL=165,178;MMQ=60,60;MPOS=36;NALOD=1.64;NLOD=12.64;POPAF=6.00;TLOD=3.33	GT:AD:AF:DP:F1R2:F2R1:SB	0/1:229,3:0.022:232:85,2:78,1:91,138,1,2	0/0:52,0:0.022:52:10,0:29,0:27,25,0,0
13	28602382	.	G	T	.	weak_evidence	AS_FilterStatus=weak_evidence;AS_SB_TABLE=42,131|2,2;DP=183;ECNT=1;GERMQ=93;MBQ=43,20;MFRL=180,143;MMQ=60,60;MPOS=29;NALOD=1.61;NLOD=11.74;POPAF=6.00;TLOD=3.91	GT:AD:AF:DP:F1R2:F2R1:SB	0/1:129,4:0.025:133:23,2:11,0:33,96,2,2	0/0:44,0:0.024:44:3,0:6,0:9,35,0,0
13	28608257	.	T	TCATATTCATATTCTCTGAAATCAACGTAGAAGTACTCATTATCTGAGGAGCCGGTCA	.	PASS	AS_FilterStatus=SITE;AS_SB_TABLE=25,15|19,17;DP=183;ECNT=1;GERMQ=73;MBQ=35,34;MFRL=211,191;MMQ=60,29;MPOS=17;NALOD=1.56;NLOD=7.76;POPAF=6.00;RPA=1,2;RU=CATATTCATATTCTCTGAAATCAACGTAGAAGTACTCATTATCTGAGGAGCCGGTCA;STR;STRQ=93;TLOD=135.09	GT:AD:AF:DP:F1R2:F2R1:SB	0/1:19,36:0.545:55:15,1:13,6:13,6,19,17	0/0:21,0:0.029:21:6,0:16,0

In [81]:
!zcat WP190.filtered.vcf.gz | grep -v "^#" | wc -l

5


In [82]:
!zcat WP190.filtered.vcf.gz | grep "^#CHROM"


#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	WP190	WP191


In [83]:
import gzip
from collections import Counter

contagem = Counter()

with gzip.open("WP190.filtered.vcf.gz", "rt") as f:
    for linha in f:
        if linha.startswith("#"):
            continue
        colunas = linha.strip().split("\t")
        contagem[colunas[6]] += 1

print("Variantes por classificação:")
for filtro, quantidade in contagem.items():
    print(f"{filtro}: {quantidade}")

Variantes por classificação:
weak_evidence: 2
PASS: 2
germline;slippage;weak_evidence: 1


In [85]:
%cd /content/somatico
!ls -lh WP190.somatic*

/content/somatico
-rw-r--r-- 1 root root 4.8K Nov 25 04:06 WP190.somatic.vcf.gz
-rw-r--r-- 1 root root   33 Nov 25 04:06 WP190.somatic.vcf.gz.stats
-rw-r--r-- 1 root root  170 Nov 25 04:06 WP190.somatic.vcf.gz.tbi


In [86]:
%cd /content/somatico

!ls -lh WP190* WP191* | head -40

/content/somatico
-rw-r--r-- 1 root root   57 Nov 25 04:10 WP190.contamination.table
-rw-r--r-- 1 root root 5.9K Nov 25 04:11 WP190.filtered.vcf.gz
-rw-r--r-- 1 root root 1.7K Nov 25 04:11 WP190.filtered.vcf.gz.filteringStats.tsv
-rw-r--r-- 1 root root  173 Nov 25 04:11 WP190.filtered.vcf.gz.tbi
-rw-r--r-- 1 root root 4.8K Nov 25 04:06 WP190.somatic.vcf.gz
-rw-r--r-- 1 root root   33 Nov 25 04:06 WP190.somatic.vcf.gz.stats
-rw-r--r-- 1 root root  170 Nov 25 04:06 WP190.somatic.vcf.gz.tbi
-rw-r--r-- 1 root root 2.1K Nov 25 04:08 WP190.table
-rw-r--r-- 1 root root 1.9K Nov 25 04:09 WP191.table


In [87]:
%cd /content/somatico

!bcftools view -H WP190.filtered.vcf.gz | cut -f1,2,4,5,7

/content/somatico
/bin/bash: line 1: bcftools: command not found


In [88]:
import gzip

arquivo = "/content/somatico/WP190.filtered.vcf.gz"

with gzip.open(arquivo, "rt") as f:
    for linha in f:
        if linha.startswith("#"):
            continue
        campos = linha.strip().split("\t")
        print(
            "Cromossomo:", campos[0],
            "| Posição:", campos[1],
            "| REF:", campos[3],
            "| ALT:", campos[4],
            "| Filtro:", campos[6]
        )

Cromossomo: 13 | Posição: 28588649 | REF: C | ALT: T | Filtro: weak_evidence
Cromossomo: 13 | Posição: 28602382 | REF: G | ALT: T | Filtro: weak_evidence
Cromossomo: 13 | Posição: 28608257 | REF: T | ALT: TCATATTCATATTCTCTGAAATCAACGTAGAAGTACTCATTATCTGAGGAGCCGGTCA | Filtro: PASS
Cromossomo: 13 | Posição: 28624322 | REF: C | ALT: A | Filtro: PASS
Cromossomo: 13 | Posição: 28635909 | REF: CA | ALT: C | Filtro: germline;slippage;weak_evidence


In [89]:
%cd /content/somatico

!find /content -name "*panel*.vcf*" -o -name "*Pon*.vcf*"

/content/somatico
/content/somatico/somatic.Pon.vcf


In [90]:
%cd /content/somatico

!head -30 somatic.Pon.vcf

/content/somatico
##fileformat=VCFv4.2
##FORMAT=<ID=AD,Number=R,Type=Integer,Description="Allelic depths for the ref and alt alleles in the order listed">
##FORMAT=<ID=AF,Number=A,Type=Float,Description="Allele fractions of alternate alleles in the tumor">
##FORMAT=<ID=DP,Number=1,Type=Integer,Description="Approximate read depth (reads with MQ=255 or with bad mates are filtered)">
##FORMAT=<ID=F1R2,Number=R,Type=Integer,Description="Count of reads in F1R2 pair orientation supporting each allele">
##FORMAT=<ID=F2R1,Number=R,Type=Integer,Description="Count of reads in F2R1 pair orientation supporting each allele">
##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype Quality">
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##FORMAT=<ID=PGT,Number=1,Type=String,Description="Physical phasing haplotype information, describing how the alternate alleles are phased in relation to one another; will always be heterozygous and is not intended to describe called alleles">
##FOR

In [91]:
!grep -v "^#" somatic.Pon.vcf | head -5

9	5026771	.	T	A	.	.	AS_SB_TABLE=0,0|0,0;DP=2;ECNT=1;MBQ=0,20;MFRL=0,128;MMQ=60,60;MPOS=21;POPAF=7.30;TLOD=4.58	GT:AD:AF:DP:F1R2:F2R1:SB	0/1:0,2:0.667:2:0,0:0,1:0,0,1,1
9	5032460	.	C	A	.	.	AS_SB_TABLE=0,0|0,0;DP=1;ECNT=1;MBQ=0,29;MFRL=0,147;MMQ=60,60;MPOS=11;POPAF=7.30;TLOD=3.08	GT:AD:AF:DP:F1R2:F2R1:SB	0/1:0,1:0.667:1:0,0:0,1:0,0,1,0
9	5032715	.	G	T	.	.	AS_SB_TABLE=0,0|0,0;DP=2;ECNT=1;MBQ=0,32;MFRL=0,148;MMQ=60,60;MPOS=12;POPAF=7.30;TLOD=5.91	GT:AD:AF:DP:F1R2:F2R1:SB	0/1:0,2:0.667:2:0,0:0,1:0,0,1,1
9	5033052	.	G	A	.	.	AS_SB_TABLE=0,0|2,1;DP=3;ECNT=1;MBQ=0,20;MFRL=0,87;MMQ=60,60;MPOS=24;POPAF=7.30;TLOD=7.70	GT:AD:AF:DP:F1R2:F2R1:SB	0/1:0,3:0.750:3:0,2:0,0:0,0,2,1
9	5035558	.	G	A	.	.	AS_SB_TABLE=1,2|0,1;DP=4;ECNT=3;MBQ=20,30;MFRL=110,0;MMQ=60,60;MPOS=26;POPAF=7.30;TLOD=3.42	GT:AD:AF:DP:F1R2:F2R1:PGT:PID:PS:SB	0|1:3,1:0.400:4:0,1:2,0:0|1:5035558_G_A:5035558:1,2,0,1


In [92]:
!find /content -type f -iname "*Mutect2*"

/content/somatico/gatk-4.2.2.0/scripts/mutect2_wdl/mutect2_pon.wdl
/content/somatico/gatk-4.2.2.0/scripts/mutect2_wdl/mutect2_multi_sample.wdl
/content/somatico/gatk-4.2.2.0/scripts/mutect2_wdl/mutect2.wdl
/content/somatico/gatk-4.2.2.0/gatkdoc/org_broadinstitute_hellbender_tools_walkers_mutect_Mutect2.json
/content/somatico/gatk-4.2.2.0/gatkdoc/org_broadinstitute_hellbender_tools_walkers_mutect_Mutect2.html


In [93]:
!grep -n -i -E 'Mutect2-exome-panel|panel.of.normals|PoN' README.md

303:## Panel of Normal (PoN)
306:# Panel of Normal (PoN)
308:GATK Best Practices - Exome PoN
314:wget -c https://storage.googleapis.com/gatk-best-practices/somatic-b37/Mutect2-exome-panel.vcf
320:wget -c https://storage.googleapis.com/gatk-best-practices/somatic-b37/Mutect2-exome-panel.vcf.idx
330:  --panel-of-normals Mutect2-exome-panel.vcf \
332:  -O WP190.somatic.pon.vcf.gz
341:	-O WP190.contamination.pon.table
349:	-V WP190.somatic.pon.vcf.gz \
350:	--contamination-table WP190.contamination.pon.table \
351:	-O WP190.filtered.pon.vcf.gz
443:* Com PoN:
445:sh run.chr13-chr19.pon.sh tumor_JAK2.bam normal_JAK2.bam WP043 WP044 
446:sh run.chr13-chr19.pon.sh tumor_wp017.bam tumor_wp018.bam WP017 WP018 
447:sh run.chr13-chr19.pon.sh tumor_wp190.bam tumor_wp191.bam WP190 WP191 
452:**Código Fonte (run.chr13-chr19.pon.sh e run.chr13-chr19.sh)**
514:pon="Mutect2-exome-panel.vcf"
525:	--panel-of-normals $pon \
526:	-O $id_tumor.somatic.pon.vcf.gz \
534:	-O $id_tumor.pon.table
538:	-I $id_tumo

In [94]:
%cd /content/somatico

!wget -c https://storage.googleapis.com/gatk-best-practices/somatic-b37/Mutect2-exome-panel.vcf

!wget -c https://storage.googleapis.com/gatk-best-practices/somatic-b37/Mutect2-exome-panel.vcf.idx

/content/somatico
--2026-11-25 04:27:36--  https://storage.googleapis.com/gatk-best-practices/somatic-b37/Mutect2-exome-panel.vcf
Resolving storage.googleapis.com (storage.googleapis.com)... 173.194.210.207, 108.177.12.207, 74.125.26.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|173.194.210.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10053980 (9.6M) [text/directory]
Saving to: ‘Mutect2-exome-panel.vcf’

Mutect2-exome-panel 100%[===================>]   9.59M  --.-KB/s    in 0.1s    

2026-11-25 04:27:36 (97.3 MB/s) - ‘Mutect2-exome-panel.vcf’ saved [10053980/10053980]

--2026-11-25 04:27:36--  https://storage.googleapis.com/gatk-best-practices/somatic-b37/Mutect2-exome-panel.vcf.idx
Resolving storage.googleapis.com (storage.googleapis.com)... 173.194.210.207, 108.177.12.207, 74.125.26.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|173.194.210.207|:443... connected.
HTTP request sent, awaiting response... 

In [95]:
!ls -lh Mutect2-exome-panel.vcf*

-rw-r--r-- 1 root root 9.6M Nov  6  2018 Mutect2-exome-panel.vcf
-rw-r--r-- 1 root root 8.6M Nov  6  2018 Mutect2-exome-panel.vcf.idx


In [96]:
%cd /content/somatico

!grep '^##contig' Mutect2-exome-panel.vcf | head -25

/content/somatico
##contig=<ID=1,length=249250621>
##contig=<ID=2,length=243199373>
##contig=<ID=3,length=198022430>
##contig=<ID=4,length=191154276>
##contig=<ID=5,length=180915260>
##contig=<ID=6,length=171115067>
##contig=<ID=7,length=159138663>
##contig=<ID=8,length=146364022>
##contig=<ID=9,length=141213431>
##contig=<ID=10,length=135534747>
##contig=<ID=11,length=135006516>
##contig=<ID=12,length=133851895>
##contig=<ID=13,length=115169878>
##contig=<ID=14,length=107349540>
##contig=<ID=15,length=102531392>
##contig=<ID=16,length=90354753>
##contig=<ID=17,length=81195210>
##contig=<ID=18,length=78077248>
##contig=<ID=19,length=59128983>
##contig=<ID=20,length=63025520>
##contig=<ID=21,length=48129895>
##contig=<ID=22,length=51304566>
##contig=<ID=X,length=155270560>
##contig=<ID=Y,length=59373566>
##contig=<ID=MT,length=16569>


In [97]:
!cat hg19_chr13_chr19.dict

@HD	VN:1.6
@SQ	SN:13	LN:115169878	M5:283f8d7892baa81b510a015719ca7b0b	UR:file:/content/somatico/hg19_chr13_chr19.fa
@SQ	SN:19	LN:59128983	M5:1aacd71f30db8e561810913e0b72636d	UR:file:/content/somatico/hg19_chr13_chr19.fa


In [98]:
%cd /content/somatico

!grep -v '^#' Mutect2-exome-panel.vcf | awk '$1=="13" || $1=="19"' | head -10

/content/somatico
13	19747991	.	C	T	.	PASS	SOMATIC
13	19748038	.	C	T	.	PASS	SOMATIC
13	19748048	.	G	A	.	PASS	SOMATIC
13	19748305	.	G	A	.	PASS	SOMATIC
13	19748344	.	C	T	.	PASS	SOMATIC
13	19751032	.	G	T	.	PASS	SOMATIC
13	19751058	.	C	T	.	PASS	SOMATIC
13	19751106	.	G	A	.	PASS	SOMATIC
13	19751157	.	A	G	.	PASS	SOMATIC
13	19751196	.	G	A	.	PASS	SOMATIC


In [99]:
%cd /content/somatico

!sed -n '510,550p' run.chr13-chr19.pon.sh

/content/somatico


In [100]:
%cd /content/somatico

!ls -lh *pon.sh

/content/somatico
-rw-r--r-- 1 root root 882 Nov 25 02:10 run.chr13-chr19.pon.sh


In [102]:
!sed -n '510,550p' run.chr13-chr19.pon.sh

In [103]:
%cd /content/somatico

!cat run.chr13-chr19.pon.sh

/content/somatico
#####
# rodando o par wp190 (tumor) wp191 (normal)
#####

# variaveis fixas
gnomad="af-only-gnomad-chr13-chr19.vcf.gz"
interval="hg19.interval_list"
genome="hg19.fa"
pon="Mutect2-exome-panel.vcf"

# bam e ids 
tumor=$1
id_tumor=$3

# rodando mutect2
./gatk-4.2.2.0/gatk Mutect2 \
	-R $genome \
	-I $tumor \
	--germline-resource $gnomad \
	--panel-of-normals $pon \
	-O $id_tumor.somatic.pon.vcf.gz \
	-L $interval

# getPileup tumor
./gatk-4.2.2.0/gatk GetPileupSummaries \
	-I $tumor \
	-V $gnomad \
	-L $interval \
	-O $id_tumor.pon.table

# CalculateContamination
./gatk-4.2.2.0/gatk CalculateContamination \
	-I $id_tumor.pon.table \
	-O $id_tumor.contamination.pon.table

# FilterMutectCalls
./gatk-4.2.2.0/gatk FilterMutectCalls \
	-R $genome \
	-V $id_tumor.somatic.pon.vcf.gz \
	--contamination-table $id_tumor.contamination.pon.table \
	-O $id_tumor.filtered.pon.vcf.gz





In [104]:
%cd /content/somatico

!ls -lh hg19.fa* hg19.interval_list hg19_chr13_chr19.fa* af-only-gnomad-chr13-chr19.vcf.gz* Mutect2-exome-panel.vcf*

/content/somatico
-rw-r--r-- 1 root root 124K Nov 25 02:10 af-only-gnomad-chr13-chr19.vcf.gz
-rw-r--r-- 1 root root  255 Nov 25 02:10 af-only-gnomad-chr13-chr19.vcf.gz.tbi
-rw-r--r-- 1 root root 170M Nov 25 04:02 hg19_chr13_chr19.fa
-rw-r--r-- 1 root root   49 Nov 25 04:02 hg19_chr13_chr19.fa.fai
-rw-r--r-- 1 root root 3.0G Nov 25 02:47 hg19.fa
-rw-r--r-- 1 root root 3.5K Nov 25 02:47 hg19.fa.fai
-rw-r--r-- 1 root root 905M Aug 21  2018 hg19.fa.gz
-rw-r--r-- 1 root root  26K Nov 25 03:41 hg19.interval_list
-rw-r--r-- 1 root root 9.6M Nov  6  2018 Mutect2-exome-panel.vcf
-rw-r--r-- 1 root root 8.6M Nov  6  2018 Mutect2-exome-panel.vcf.idx


In [105]:
!ls -lh tumor_wp190.bam* normal_wp191.bam*

ls: cannot access 'normal_wp191.bam*': No such file or directory
-rw-r--r-- 1 root root 1.9M Nov 25 02:10  tumor_wp190.bam
-rw-r--r-- 1 root root  15K Nov 25 02:10  tumor_wp190.bam.bai


In [106]:
%cd /content/somatico

!head -20 hg19.interval_list

/content/somatico
@HD	VN:1.6	SO:coordinate
@SQ	SN:chr1	LN:249250621	M5:1b22b98cdeb4a9304cb5d48026a85128	UR:file:/content/somatico/hg19.fa
@SQ	SN:chr2	LN:243199373	M5:a0d9851da00400dec1098a9255ac712e	UR:file:/content/somatico/hg19.fa
@SQ	SN:chr3	LN:198022430	M5:641e4338fa8d52a5b781bd2a2c08d3c3	UR:file:/content/somatico/hg19.fa
@SQ	SN:chr4	LN:191154276	M5:23dccd106897542ad87d2765d28a19a1	UR:file:/content/somatico/hg19.fa
@SQ	SN:chr5	LN:180915260	M5:0740173db9ffd264d728f32784845cd7	UR:file:/content/somatico/hg19.fa
@SQ	SN:chr6	LN:171115067	M5:1d3a93a248d92a729ee764823acbbc6b	UR:file:/content/somatico/hg19.fa
@SQ	SN:chr7	LN:159138663	M5:618366e953d6aaad97dbe4777c29375e	UR:file:/content/somatico/hg19.fa
@SQ	SN:chrX	LN:155270560	M5:7e0e2e580297b7764e31dbc80c2540dd	UR:file:/content/somatico/hg19.fa
@SQ	SN:chr8	LN:146364022	M5:96f514a9929e410c6651697bded59aec	UR:file:/content/somatico/hg19.fa
@SQ	SN:chr9	LN:141213431	M5:3e273117f15e0a400f01055d9f393768	UR:file:/content/somatico/hg19.fa
@SQ	SN:

In [107]:
!ls -lh *.bam*

-rw-r--r-- 1 root root 574K Nov 25 02:24 'normal_JAK2 (1).bam'
-rw-r--r-- 1 root root 574K Nov 25 02:10  normal_JAK2.bam
-rw-r--r-- 1 root root 2.9K Nov 25 02:35  normal_JAK2.bam.bai
-rw-r--r-- 1 root root 997K Nov 25 02:34  tumor_JAK2.bam
-rw-r--r-- 1 root root 2.9K Nov 25 02:34  tumor_JAK2.bam.bai
-rw-r--r-- 1 root root 791K Nov 25 02:10  tumor_wp017.bam
-rw-r--r-- 1 root root 6.5K Nov 25 02:10  tumor_wp017.bam.bai
-rw-r--r-- 1 root root 384K Nov 25 02:10  tumor_wp018.bam
-rw-r--r-- 1 root root 6.5K Nov 25 02:10  tumor_wp018.bam.bai
-rw-r--r-- 1 root root 1.9M Nov 25 02:10  tumor_wp190.bam
-rw-r--r-- 1 root root  15K Nov 25 02:10  tumor_wp190.bam.bai
-rw-r--r-- 1 root root 605K Nov 25 02:10  tumor_wp191.bam
-rw-r--r-- 1 root root  15K Nov 25 02:10  tumor_wp191.bam.bai


In [111]:
%%bash
cd /content/somatico

echo "=== INTERVALOS ==="
grep -E 'chr13|chr19|^13|^19' hg19.interval_list | head -20

echo "=== CONTIGS DO PON ==="
grep '^##contig' Mutect2-exome-panel.vcf | grep -E 'ID=chr?(13|19)[,>]'

echo "=== CONTIGS DO GNOMAD ==="
zgrep '^##contig' af-only-gnomad-chr13-chr19.vcf.gz

=== INTERVALOS ===
@SQ	SN:chr13	LN:115169878	M5:283f8d7892baa81b510a015719ca7b0b	UR:file:/content/somatico/hg19.fa
@SQ	SN:chr19	LN:59128983	M5:1aacd71f30db8e561810913e0b72636d	UR:file:/content/somatico/hg19.fa
@SQ	SN:chr19_gl000209_random	LN:159169	M5:f40598e2a5a6b26e84a3775e0d1e2c81	UR:file:/content/somatico/hg19.fa
@SQ	SN:chr19_gl000208_random	LN:92689	M5:aa81be49bf3fe63a79bdc6a6f279abf6	UR:file:/content/somatico/hg19.fa
chr13	19020001	86760324	+	ACGTmer
chr13	86910325	112353994	+	ACGTmer
chr13	112503995	114325993	+	ACGTmer
chr13	114425994	114639948	+	ACGTmer
chr13	114739949	115109878	+	ACGTmer
chr19	60001	7346004	+	ACGTmer
chr19	7396005	8687198	+	ACGTmer
chr19	8737199	20523415	+	ACGTmer
chr19	20573416	24631782	+	ACGTmer
chr19	27731783	59118983	+	ACGTmer
chr19_gl000209_random	1	159169	+	ACGTmer
chr19_gl000208_random	1	92689	+	ACGTmer
=== CONTIGS DO PON ===
=== CONTIGS DO GNOMAD ===


CalledProcessError: Command 'b'cd /content/somatico\n\necho "=== INTERVALOS ==="\ngrep -E \'chr13|chr19|^13|^19\' hg19.interval_list | head -20\n\necho "=== CONTIGS DO PON ==="\ngrep \'^##contig\' Mutect2-exome-panel.vcf | grep -E \'ID=chr?(13|19)[,>]\'\n\necho "=== CONTIGS DO GNOMAD ==="\nzgrep \'^##contig\' af-only-gnomad-chr13-chr19.vcf.gz\n'' returned non-zero exit status 1.

In [112]:
%%bash
cd /content/somatico

echo "=== CABEÇALHO DO PON ==="
head -30 Mutect2-exome-panel.vcf

echo "=== PRIMEIRAS VARIANTES DO PON ==="
grep -v '^#' Mutect2-exome-panel.vcf | head -5

echo "=== CABEÇALHO DO GNOMAD ==="
zcat af-only-gnomad-chr13-chr19.vcf.gz | head -30

echo "=== PRIMEIRAS VARIANTES DO GNOMAD ==="
zcat af-only-gnomad-chr13-chr19.vcf.gz | grep -v '^#' | head -5

=== CABEÇALHO DO PON ===
##fileformat=VCFv4.2
##INFO=<ID=SOMATIC,Number=0,Type=Flag,Description="somatic">
##Mutect Version=2.1-beta
##contig=<ID=1,length=249250621>
##contig=<ID=2,length=243199373>
##contig=<ID=3,length=198022430>
##contig=<ID=4,length=191154276>
##contig=<ID=5,length=180915260>
##contig=<ID=6,length=171115067>
##contig=<ID=7,length=159138663>
##contig=<ID=8,length=146364022>
##contig=<ID=9,length=141213431>
##contig=<ID=10,length=135534747>
##contig=<ID=11,length=135006516>
##contig=<ID=12,length=133851895>
##contig=<ID=13,length=115169878>
##contig=<ID=14,length=107349540>
##contig=<ID=15,length=102531392>
##contig=<ID=16,length=90354753>
##contig=<ID=17,length=81195210>
##contig=<ID=18,length=78077248>
##contig=<ID=19,length=59128983>
##contig=<ID=20,length=63025520>
##contig=<ID=21,length=48129895>
##contig=<ID=22,length=51304566>
##contig=<ID=X,length=155270560>
##contig=<ID=Y,length=59373566>
##contig=<ID=MT,length=16569>
##contig=<ID=GL000207.1,length=4262>
##c

In [113]:
%%bash
cd /content/somatico

echo "=== CONTIGS DO BAM WP190 ==="
samtools view -H tumor_wp190.bam | grep '^@SQ' | head -20

echo "=== CONTIGS DO BAM WP191 ==="
samtools view -H tumor_wp191.bam | grep '^@SQ' | head -20

=== CONTIGS DO BAM WP190 ===
@SQ	SN:1	LN:249250621
@SQ	SN:2	LN:243199373
@SQ	SN:3	LN:198022430
@SQ	SN:4	LN:191154276
@SQ	SN:5	LN:180915260
@SQ	SN:6	LN:171115067
@SQ	SN:7	LN:159138663
@SQ	SN:8	LN:146364022
@SQ	SN:9	LN:141213431
@SQ	SN:10	LN:135534747
@SQ	SN:11	LN:135006516
@SQ	SN:12	LN:133851895
@SQ	SN:13	LN:115169878
@SQ	SN:14	LN:107349540
@SQ	SN:15	LN:102531392
@SQ	SN:16	LN:90354753
@SQ	SN:17	LN:81195210
@SQ	SN:18	LN:78077248
@SQ	SN:19	LN:59128983
@SQ	SN:20	LN:63025520
=== CONTIGS DO BAM WP191 ===
@SQ	SN:1	LN:249250621
@SQ	SN:2	LN:243199373
@SQ	SN:3	LN:198022430
@SQ	SN:4	LN:191154276
@SQ	SN:5	LN:180915260
@SQ	SN:6	LN:171115067
@SQ	SN:7	LN:159138663
@SQ	SN:8	LN:146364022
@SQ	SN:9	LN:141213431
@SQ	SN:10	LN:135534747
@SQ	SN:11	LN:135006516
@SQ	SN:12	LN:133851895
@SQ	SN:13	LN:115169878
@SQ	SN:14	LN:107349540
@SQ	SN:15	LN:102531392
@SQ	SN:16	LN:90354753
@SQ	SN:17	LN:81195210
@SQ	SN:18	LN:78077248
@SQ	SN:19	LN:59128983
@SQ	SN:20	LN:63025520


In [114]:
%%bash
cd /content/somatico

echo "=== CABEÇALHO DA REFERÊNCIA REDUZIDA ==="
head -4 hg19_chr13_chr19.fa

echo "=== ÍNDICE ==="
cat hg19_chr13_chr19.fa.fai

echo "=== DICIONÁRIO ==="
ls -lh hg19_chr13_chr19.dict
cat hg19_chr13_chr19.dict

echo "=== INTERVALOS ==="
head -8 hg19.interval_list

=== CABEÇALHO DA REFERÊNCIA REDUZIDA ===
>13
NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN
NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN
NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN
=== ÍNDICE ===
13	115169878	4	50	51
19	59128983	117473284	50	51
=== DICIONÁRIO ===
-rw-r--r-- 1 root root 220 Nov 25 04:03 hg19_chr13_chr19.dict
@HD	VN:1.6
@SQ	SN:13	LN:115169878	M5:283f8d7892baa81b510a015719ca7b0b	UR:file:/content/somatico/hg19_chr13_chr19.fa
@SQ	SN:19	LN:59128983	M5:1aacd71f30db8e561810913e0b72636d	UR:file:/content/somatico/hg19_chr13_chr19.fa
=== INTERVALOS ===
@HD	VN:1.6	SO:coordinate
@SQ	SN:chr1	LN:249250621	M5:1b22b98cdeb4a9304cb5d48026a85128	UR:file:/content/somatico/hg19.fa
@SQ	SN:chr2	LN:243199373	M5:a0d9851da00400dec1098a9255ac712e	UR:file:/content/somatico/hg19.fa
@SQ	SN:chr3	LN:198022430	M5:641e4338fa8d52a5b781bd2a2c08d3c3	UR:file:/content/somatico/hg19.fa
@SQ	SN:chr4	LN:191154276	M5:23dccd106897542ad87d2765d28a19a1	UR:file:/content/somatico/hg19.fa
@SQ	SN:chr5	L

In [115]:
%%bash
cd /content/somatico

echo "=== CHR13: PRIMEIRAS 50 BASES ==="
samtools faidx hg19.fa chr13:1-50

echo "=== CHR19: PRIMEIRAS 50 BASES ==="
samtools faidx hg19.fa chr19:1-50

=== CHR13: PRIMEIRAS 50 BASES ===
>chr13:1-50
NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN
=== CHR19: PRIMEIRAS 50 BASES ===
>chr19:1-50
NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN


In [116]:
%%bash
cd /content/somatico

echo "=== HG19 ORIGINAL: CHR13 ==="
samtools faidx hg19.fa chr13:19020001-19020050

echo "=== HG19 ORIGINAL: CHR19 ==="
samtools faidx hg19.fa chr19:60001-60050

echo "=== REFERÊNCIA REDUZIDA: 13 ==="
samtools faidx hg19_chr13_chr19.fa 13:19020001-19020050

echo "=== REFERÊNCIA REDUZIDA: 19 ==="
samtools faidx hg19_chr13_chr19.fa 19:60001-60050

=== HG19 ORIGINAL: CHR13 ===
>chr13:19020001-19020050
GAATTCAACTGCCTCCCTGGCCTTTCCCTGCCAATCTATGTGCCCCAGCA
=== HG19 ORIGINAL: CHR19 ===
>chr19:60001-60050
GATCACAGAGGCTGGGCTGCTCCCCACCCTCTGCACACCTCCTGCTTCTA
=== REFERÊNCIA REDUZIDA: 13 ===
>13:19020001-19020050
GAATTCAACTGCCTCCCTGGCCTTTCCCTGCCAATCTATGTGCCCCAGCA
=== REFERÊNCIA REDUZIDA: 19 ===
>19:60001-60050
GATCACAGAGGCTGGGCTGCTCCCCACCCTCTGCACACCTCCTGCTTCTA


In [117]:
%%bash
cd /content/somatico

awk '
BEGIN { FS=OFS="\t" }
/^@/ {
    if ($1=="@SQ") {
        sub("chr","",$2)
        for(i=1;i<=NF;i++) {
            if ($i ~ /UR:file:/) $i="UR:file:/content/somatico/hg19_chr13_chr19.fa"
        }
    }
    print
    next
}
{
    if ($1=="chr13" || $1=="chr19") {
        sub("chr","",$1)
        print
    }
}
' hg19.interval_list > hg19_chr13_chr19.interval_list

head -8 hg19_chr13_chr19.interval_list

@HD	VN:1.6	SO:coordinate
@SQ	SN:1	LN:249250621	M5:1b22b98cdeb4a9304cb5d48026a85128	UR:file:/content/somatico/hg19_chr13_chr19.fa
@SQ	SN:2	LN:243199373	M5:a0d9851da00400dec1098a9255ac712e	UR:file:/content/somatico/hg19_chr13_chr19.fa
@SQ	SN:3	LN:198022430	M5:641e4338fa8d52a5b781bd2a2c08d3c3	UR:file:/content/somatico/hg19_chr13_chr19.fa
@SQ	SN:4	LN:191154276	M5:23dccd106897542ad87d2765d28a19a1	UR:file:/content/somatico/hg19_chr13_chr19.fa
@SQ	SN:5	LN:180915260	M5:0740173db9ffd264d728f32784845cd7	UR:file:/content/somatico/hg19_chr13_chr19.fa
@SQ	SN:6	LN:171115067	M5:1d3a93a248d92a729ee764823acbbc6b	UR:file:/content/somatico/hg19_chr13_chr19.fa
@SQ	SN:7	LN:159138663	M5:618366e953d6aaad97dbe4777c29375e	UR:file:/content/somatico/hg19_chr13_chr19.fa


In [118]:
%%bash
cd /content/somatico

awk '
BEGIN { FS=OFS="\t" }
/^@HD/ { print; next }
/^@SQ/ {
    if ($2=="SN:chr13" || $2=="SN:chr19") {
        sub("SN:chr","SN:",$2)
        for (i=1; i<=NF; i++)
            if ($i ~ /UR:file:/)
                $i="UR:file:/content/somatico/hg19_chr13_chr19.fa"
        print
    }
    next
}
/^@/ { print; next }
$1=="chr13" || $1=="chr19" {
    sub("chr","",$1)
    print
}
' hg19.interval_list > hg19_chr13_chr19.interval_list

echo "=== CABEÇALHO ==="
head -5 hg19_chr13_chr19.interval_list

echo "=== REGIÕES ==="
grep -v '^@' hg19_chr13_chr19.interval_list | head -5

echo "=== CONTAGEM POR CROMOSSOMO ==="
grep -v '^@' hg19_chr13_chr19.interval_list | cut -f1 | sort | uniq -c

=== CABEÇALHO ===
@HD	VN:1.6	SO:coordinate
@SQ	SN:13	LN:115169878	M5:283f8d7892baa81b510a015719ca7b0b	UR:file:/content/somatico/hg19_chr13_chr19.fa
@SQ	SN:19	LN:59128983	M5:1aacd71f30db8e561810913e0b72636d	UR:file:/content/somatico/hg19_chr13_chr19.fa
13	19020001	86760324	+	ACGTmer
13	86910325	112353994	+	ACGTmer
=== REGIÕES ===
13	19020001	86760324	+	ACGTmer
13	86910325	112353994	+	ACGTmer
13	112503995	114325993	+	ACGTmer
13	114425994	114639948	+	ACGTmer
13	114739949	115109878	+	ACGTmer
=== CONTAGEM POR CROMOSSOMO ===
      5 13
      5 19


In [119]:
%%bash
cd /content/somatico

ls -lh hg19_chr13_chr19.fa*
ls -lh hg19_chr13_chr19.dict

-rw-r--r-- 1 root root 170M Nov 25 04:02 hg19_chr13_chr19.fa
-rw-r--r-- 1 root root   49 Nov 25 04:02 hg19_chr13_chr19.fa.fai
-rw-r--r-- 1 root root 220 Nov 25 04:03 hg19_chr13_chr19.dict


In [120]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk ValidateVariants \
  -R hg19_chr13_chr19.fa \
  -V Mutect2-exome-panel.vcf \
  -L hg19_chr13_chr19.interval_list \
  --validation-type-to-exclude ALL

[0.013s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.013s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar ValidateVariants -R hg19_chr13_chr19.fa -V Mutect2-exome-panel.vcf -L hg19_chr13_chr19.interval_list --validation-type-to-exclude ALL
04:43:54.621 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
04:43:55.047 INFO  ValidateVariants - ------------------------------------------------------------
04:43:55.048 INFO  ValidateVariants - The Genome Analysis Toolkit (GATK) v4.2.2.0
04:43:55.048 INFO  ValidateVariants - For support and documentation go to https://software.broadinstitute.org/gatk/
04:43:55.048 INFO  ValidateVariants - Executing as root@97750

In [121]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk Mutect2 \
  -R hg19_chr13_chr19.fa \
  -I tumor_wp190.bam \
  --germline-resource af-only-gnomad-chr13-chr19.vcf.gz \
  --panel-of-normals Mutect2-exome-panel.vcf \
  -O WP190.somatic.pon.vcf.gz \
  -L hg19_chr13_chr19.interval_list

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Tool returned:
SUCCESS


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar Mutect2 -R hg19_chr13_chr19.fa -I tumor_wp190.bam --germline-resource af-only-gnomad-chr13-chr19.vcf.gz --panel-of-normals Mutect2-exome-panel.vcf -O WP190.somatic.pon.vcf.gz -L hg19_chr13_chr19.interval_list
04:46:58.632 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
04:46:58.917 INFO  Mutect2 - ------------------------------------------------------------
04:46:58.918 INFO  Mutect2 - The Genome Analysis Toolkit (GATK) v4.2.2.0
04:46:58.919 INFO  Mutect2 - For support and documentation go to https://software.broadinstitute.org/gatk/
04:46:58.919 

In [122]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk GetPileupSummaries \
  -I tumor_wp190.bam \
  -V af-only-gnomad-chr13-chr19.vcf.gz \
  -L hg19_chr13_chr19.interval_list \
  -O WP190.pon.table

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Tool returned:
SUCCESS


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar GetPileupSummaries -I tumor_wp190.bam -V af-only-gnomad-chr13-chr19.vcf.gz -L hg19_chr13_chr19.interval_list -O WP190.pon.table
04:49:27.458 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
04:49:27.735 INFO  GetPileupSummaries - ------------------------------------------------------------
04:49:27.735 INFO  GetPileupSummaries - The Genome Analysis Toolkit (GATK) v4.2.2.0
04:49:27.736 INFO  GetPileupSummaries - For support and documentation go to https://software.broadinstitute.org/gatk/
04:49:27.736 INFO  GetPileupSummaries - Executing as root@977

In [123]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk CalculateContamination \
  -I WP190.pon.table \
  -O WP190.contamination.pon.table

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Tool returned:
SUCCESS


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar CalculateContamination -I WP190.pon.table -O WP190.contamination.pon.table
04:50:28.631 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
04:50:28.942 INFO  CalculateContamination - ------------------------------------------------------------
04:50:28.944 INFO  CalculateContamination - The Genome Analysis Toolkit (GATK) v4.2.2.0
04:50:28.944 INFO  CalculateContamination - For support and documentation go to https://software.broadinstitute.org/gatk/
04:50:28.945 INFO  CalculateContamination - Executing as root@97750e0669bd on Linux v6.6.159+ amd64
04

In [124]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk FilterMutectCalls \
  -R hg19_chr13_chr19.fa \
  -V WP190.somatic.pon.vcf.gz \
  --contamination-table WP190.contamination.pon.table \
  -O WP190.filtered.pon.vcf.gz

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar FilterMutectCalls -R hg19_chr13_chr19.fa -V WP190.somatic.pon.vcf.gz --contamination-table WP190.contamination.pon.table -O WP190.filtered.pon.vcf.gz
04:51:22.087 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
04:51:22.744 INFO  FilterMutectCalls - ------------------------------------------------------------
04:51:22.744 INFO  FilterMutectCalls - The Genome Analysis Toolkit (GATK) v4.2.2.0
04:51:22.745 INFO  FilterMutectCalls - For support and documentation go to https://software.broadinstitute.org/gatk/
04:51:22.748 INFO  FilterMutectCalls - Exe

In [125]:
import gzip
from collections import Counter

arquivos = {
    "PAIRED": "/content/somatico/WP190.filtered.vcf.gz",
    "PON": "/content/somatico/WP190.filtered.pon.vcf.gz"
}

resultados = {}

for modo, arquivo in arquivos.items():
    variantes = {}

    with gzip.open(arquivo, "rt") as f:
        for linha in f:
            if linha.startswith("#"):
                continue

            c = linha.strip().split("\t")
            crom = c[0].replace("chr", "")
            chave = (crom, c[1], c[3], c[4])

            variantes[chave] = c[6]

    resultados[modo] = variantes

    print(f"\n===== {modo} =====")
    print("Total de variantes:", len(variantes))
    print("Classificação FILTER:", dict(Counter(variantes.values())))

    for chave, filtro in variantes.items():
        print(
            f"Chr {chave[0]} | Posição {chave[1]} | "
            f"REF {chave[2]} | ALT {chave[3]} | FILTER {filtro}"
        )

paired = set(resultados["PAIRED"])
pon = set(resultados["PON"])

print("\n===== COMPARAÇÃO =====")
print("Presentes nos dois métodos:", len(paired & pon))
print("Exclusivas do PAIRED:", len(paired - pon))
print("Exclusivas do PON:", len(pon - paired))

print("\nVariantes exclusivas do PAIRED:")
for v in sorted(paired - pon):
    print(v, "| FILTER:", resultados["PAIRED"][v])

print("\nVariantes exclusivas do PON:")
for v in sorted(pon - paired):
    print(v, "| FILTER:", resultados["PON"][v])

print("\nVariantes compartilhadas e seus filtros:")
for v in sorted(paired & pon):
    print(
        v,
        "| PAIRED:", resultados["PAIRED"][v],
        "| PON:", resultados["PON"][v]
    )


===== PAIRED =====
Total de variantes: 5
Classificação FILTER: {'weak_evidence': 2, 'PASS': 2, 'germline;slippage;weak_evidence': 1}
Chr 13 | Posição 28588649 | REF C | ALT T | FILTER weak_evidence
Chr 13 | Posição 28602382 | REF G | ALT T | FILTER weak_evidence
Chr 13 | Posição 28608257 | REF T | ALT TCATATTCATATTCTCTGAAATCAACGTAGAAGTACTCATTATCTGAGGAGCCGGTCA | FILTER PASS
Chr 13 | Posição 28624322 | REF C | ALT A | FILTER PASS
Chr 13 | Posição 28635909 | REF CA | ALT C | FILTER germline;slippage;weak_evidence

===== PON =====
Total de variantes: 35
Classificação FILTER: {'germline;weak_evidence': 3, 'map_qual;weak_evidence': 1, 'weak_evidence': 5, 'germline;panel_of_normals': 6, 'germline': 9, 'PASS': 3, 'germline;map_qual': 1, 'germline;slippage;weak_evidence': 1, 'clustered_events;germline;haplotype;map_qual;weak_evidence': 1, 'clustered_events;haplotype;map_qual': 2, 'germline;multiallelic': 1, 'germline;haplotype': 1, 'haplotype': 1}
Chr 13 | Posição 28580716 | REF G | ALT A | FI

In [126]:
%%bash
cd /content/somatico
ls -lh tumor_wp017.bam tumor_wp043.bam

-rw-r--r-- 1 root root 791K Nov 25 02:10 tumor_wp017.bam


ls: cannot access 'tumor_wp043.bam': No such file or directory


CalledProcessError: Command 'b'cd /content/somatico\nls -lh tumor_wp017.bam tumor_wp043.bam\n'' returned non-zero exit status 2.

In [127]:
%%bash
cd /content/somatico
ls -lh *.bam

-rw-r--r-- 1 root root 574K Nov 25 02:24 normal_JAK2 (1).bam
-rw-r--r-- 1 root root 574K Nov 25 02:10 normal_JAK2.bam
-rw-r--r-- 1 root root 997K Nov 25 02:34 tumor_JAK2.bam
-rw-r--r-- 1 root root 791K Nov 25 02:10 tumor_wp017.bam
-rw-r--r-- 1 root root 384K Nov 25 02:10 tumor_wp018.bam
-rw-r--r-- 1 root root 1.9M Nov 25 02:10 tumor_wp190.bam
-rw-r--r-- 1 root root 605K Nov 25 02:10 tumor_wp191.bam


In [128]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk Mutect2 \
  -R hg19_chr13_chr19.fa \
  -I tumor_wp017.bam \
  --germline-resource af-only-gnomad-chr13-chr19.vcf.gz \
  --panel-of-normals Mutect2-exome-panel.vcf \
  -O WP017.somatic.pon.vcf.gz \
  -L hg19_chr13_chr19.interval_list

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Tool returned:
SUCCESS


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar Mutect2 -R hg19_chr13_chr19.fa -I tumor_wp017.bam --germline-resource af-only-gnomad-chr13-chr19.vcf.gz --panel-of-normals Mutect2-exome-panel.vcf -O WP017.somatic.pon.vcf.gz -L hg19_chr13_chr19.interval_list
04:53:52.940 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
04:53:53.194 INFO  Mutect2 - ------------------------------------------------------------
04:53:53.195 INFO  Mutect2 - The Genome Analysis Toolkit (GATK) v4.2.2.0
04:53:53.195 INFO  Mutect2 - For support and documentation go to https://software.broadinstitute.org/gatk/
04:53:53.196 

In [129]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk GetPileupSummaries \
  -I tumor_wp017.bam \
  -V af-only-gnomad-chr13-chr19.vcf.gz \
  -L hg19_chr13_chr19.interval_list \
  -O WP017.pon.table

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Tool returned:
SUCCESS


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar GetPileupSummaries -I tumor_wp017.bam -V af-only-gnomad-chr13-chr19.vcf.gz -L hg19_chr13_chr19.interval_list -O WP017.pon.table
04:55:19.139 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
04:55:19.418 INFO  GetPileupSummaries - ------------------------------------------------------------
04:55:19.419 INFO  GetPileupSummaries - The Genome Analysis Toolkit (GATK) v4.2.2.0
04:55:19.419 INFO  GetPileupSummaries - For support and documentation go to https://software.broadinstitute.org/gatk/
04:55:19.420 INFO  GetPileupSummaries - Executing as root@977

In [130]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk CalculateContamination \
  -I WP017.pon.table \
  -O WP017.contamination.pon.table

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Tool returned:
SUCCESS


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar CalculateContamination -I WP017.pon.table -O WP017.contamination.pon.table
04:55:47.626 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
04:55:47.913 INFO  CalculateContamination - ------------------------------------------------------------
04:55:47.914 INFO  CalculateContamination - The Genome Analysis Toolkit (GATK) v4.2.2.0
04:55:47.914 INFO  CalculateContamination - For support and documentation go to https://software.broadinstitute.org/gatk/
04:55:47.914 INFO  CalculateContamination - Executing as root@97750e0669bd on Linux v6.6.159+ amd64
04

In [131]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk FilterMutectCalls \
  -R hg19_chr13_chr19.fa \
  -V WP017.somatic.pon.vcf.gz \
  --contamination-table WP017.contamination.pon.table \
  -O WP017.filtered.pon.vcf.gz

[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar FilterMutectCalls -R hg19_chr13_chr19.fa -V WP017.somatic.pon.vcf.gz --contamination-table WP017.contamination.pon.table -O WP017.filtered.pon.vcf.gz
04:56:21.741 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
04:56:22.046 INFO  FilterMutectCalls - ------------------------------------------------------------
04:56:22.047 INFO  FilterMutectCalls - The Genome Analysis Toolkit (GATK) v4.2.2.0
04:56:22.047 INFO  FilterMutectCalls - For support and documentation go to https://software.broadinstitute.org/gatk/
04:56:22.047 INFO  FilterMutectCalls - Exe

In [132]:
import gzip

arquivo = "/content/somatico/WP017.filtered.pon.vcf.gz"

with gzip.open(arquivo, "rt") as f:
    for linha in f:
        if linha.startswith("#"):
            continue
        campos = linha.strip().split("\t")
        print(
            f"chr{campos[0]}:{campos[1]} "
            f"{campos[3]}>{campos[4]} "
            f"FILTRO={campos[6]}"
        )

chr19:13049389 T>G FILTRO=base_qual;contamination;germline;haplotype;weak_evidence
chr19:13049393 T>G FILTRO=base_qual;contamination;germline;haplotype
chr19:13049971 G>A FILTRO=contamination;weak_evidence
chr19:13053407 CTTT>C FILTRO=contamination;germline;weak_evidence
chr19:13053478 G>A FILTRO=contamination;germline;weak_evidence
chr19:13054564 AGCAGAGGCTTAAGGAGGAGGAAGAAGACAAGAAACGCAAAGAGGAGGAGGAG>A FILTRO=PASS
chr19:13054781 G>T FILTRO=germline


In [133]:
%%bash
cd /content/somatico
ls -lh WP017*

-rw-r--r-- 1 root root   73 Nov 25 04:55 WP017.contamination.pon.table
-rw-r--r-- 1 root root 5.4K Nov 25 04:56 WP017.filtered.pon.vcf.gz
-rw-r--r-- 1 root root 1.7K Nov 25 04:56 WP017.filtered.pon.vcf.gz.filteringStats.tsv
-rw-r--r-- 1 root root  129 Nov 25 04:56 WP017.filtered.pon.vcf.gz.tbi
-rw-r--r-- 1 root root  291 Nov 25 04:55 WP017.pon.table
-rw-r--r-- 1 root root 4.2K Nov 25 04:54 WP017.somatic.pon.vcf.gz
-rw-r--r-- 1 root root   32 Nov 25 04:54 WP017.somatic.pon.vcf.gz.stats
-rw-r--r-- 1 root root  129 Nov 25 04:54 WP017.somatic.pon.vcf.gz.tbi


In [134]:
%%bash
cd /content/somatico
ls -lh *wp017* *wp018* *WP017* *WP018* 2>/dev/null

-rw-r--r-- 1 root root 791K Nov 25 02:10 tumor_wp017.bam
-rw-r--r-- 1 root root 6.5K Nov 25 02:10 tumor_wp017.bam.bai
-rw-r--r-- 1 root root 384K Nov 25 02:10 tumor_wp018.bam
-rw-r--r-- 1 root root 6.5K Nov 25 02:10 tumor_wp018.bam.bai
-rw-r--r-- 1 root root   73 Nov 25 04:55 WP017.contamination.pon.table
-rw-r--r-- 1 root root 5.4K Nov 25 04:56 WP017.filtered.pon.vcf.gz
-rw-r--r-- 1 root root 1.7K Nov 25 04:56 WP017.filtered.pon.vcf.gz.filteringStats.tsv
-rw-r--r-- 1 root root  129 Nov 25 04:56 WP017.filtered.pon.vcf.gz.tbi
-rw-r--r-- 1 root root  291 Nov 25 04:55 WP017.pon.table
-rw-r--r-- 1 root root 4.2K Nov 25 04:54 WP017.somatic.pon.vcf.gz
-rw-r--r-- 1 root root   32 Nov 25 04:54 WP017.somatic.pon.vcf.gz.stats
-rw-r--r-- 1 root root  129 Nov 25 04:54 WP017.somatic.pon.vcf.gz.tbi


CalledProcessError: Command 'b'cd /content/somatico\nls -lh *wp017* *wp018* *WP017* *WP018* 2>/dev/null\n'' returned non-zero exit status 2.

In [135]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk Mutect2 \
  -R hg19_chr13_chr19.fa \
  -I tumor_wp017.bam \
  -I tumor_wp018.bam \
  -normal tumor_wp018 \
  --germline-resource af-only-gnomad-chr13-chr19.vcf.gz \
  -O WP017.somatic.vcf.gz \
  -L hg19_chr13_chr19.interval_list

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar Mutect2 -R hg19_chr13_chr19.fa -I tumor_wp017.bam -I tumor_wp018.bam -normal tumor_wp018 --germline-resource af-only-gnomad-chr13-chr19.vcf.gz -O WP017.somatic.vcf.gz -L hg19_chr13_chr19.interval_list
04:59:04.780 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
04:59:05.173 INFO  Mutect2 - ------------------------------------------------------------
04:59:05.173 INFO  Mutect2 - The Genome Analysis Toolkit (GATK) v4.2.2.0
04:59:05.174 INFO  Mutect2 - For support and documentation go to https://software.broadinstitute.org/gatk/
04:59:05.178 INFO  Mu

CalledProcessError: Command 'b'cd /content/somatico\n\n./gatk-4.2.2.0/gatk Mutect2 \\\n  -R hg19_chr13_chr19.fa \\\n  -I tumor_wp017.bam \\\n  -I tumor_wp018.bam \\\n  -normal tumor_wp018 \\\n  --germline-resource af-only-gnomad-chr13-chr19.vcf.gz \\\n  -O WP017.somatic.vcf.gz \\\n  -L hg19_chr13_chr19.interval_list\n'' returned non-zero exit status 2.

In [136]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk Mutect2 \
  -R hg19_chr13_chr19.fa \
  -I tumor_wp017.bam \
  -I tumor_wp018.bam \
  -normal WP018 \
  --germline-resource af-only-gnomad-chr13-chr19.vcf.gz \
  -O WP017.somatic.vcf.gz \
  -L hg19_chr13_chr19.interval_list

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Tool returned:
SUCCESS


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar Mutect2 -R hg19_chr13_chr19.fa -I tumor_wp017.bam -I tumor_wp018.bam -normal WP018 --germline-resource af-only-gnomad-chr13-chr19.vcf.gz -O WP017.somatic.vcf.gz -L hg19_chr13_chr19.interval_list
04:59:57.481 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
04:59:57.751 INFO  Mutect2 - ------------------------------------------------------------
04:59:57.752 INFO  Mutect2 - The Genome Analysis Toolkit (GATK) v4.2.2.0
04:59:57.753 INFO  Mutect2 - For support and documentation go to https://software.broadinstitute.org/gatk/
04:59:57.753 INFO  Mutect2 

In [137]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk GetPileupSummaries \
  -I tumor_wp017.bam \
  -V af-only-gnomad-chr13-chr19.vcf.gz \
  -L hg19_chr13_chr19.interval_list \
  -O WP017.paired.tumor.table

./gatk-4.2.2.0/gatk GetPileupSummaries \
  -I tumor_wp018.bam \
  -V af-only-gnomad-chr13-chr19.vcf.gz \
  -L hg19_chr13_chr19.interval_list \
  -O WP017.paired.normal.table

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Tool returned:
SUCCESS
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Tool returned:
SUCCESS


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar GetPileupSummaries -I tumor_wp017.bam -V af-only-gnomad-chr13-chr19.vcf.gz -L hg19_chr13_chr19.interval_list -O WP017.paired.tumor.table
05:01:30.544 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
05:01:30.845 INFO  GetPileupSummaries - ------------------------------------------------------------
05:01:30.846 INFO  GetPileupSummaries - The Genome Analysis Toolkit (GATK) v4.2.2.0
05:01:30.846 INFO  GetPileupSummaries - For support and documentation go to https://software.broadinstitute.org/gatk/
05:01:30.847 INFO  GetPileupSummaries - Executing as

In [138]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk CalculateContamination \
  -I WP017.paired.tumor.table \
  -matched WP017.paired.normal.table \
  -O WP017.paired.contamination.table

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Tool returned:
SUCCESS


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar CalculateContamination -I WP017.paired.tumor.table -matched WP017.paired.normal.table -O WP017.paired.contamination.table
05:02:18.062 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
05:02:18.345 INFO  CalculateContamination - ------------------------------------------------------------
05:02:18.345 INFO  CalculateContamination - The Genome Analysis Toolkit (GATK) v4.2.2.0
05:02:18.346 INFO  CalculateContamination - For support and documentation go to https://software.broadinstitute.org/gatk/
05:02:18.346 INFO  CalculateContamination - Executing a

In [139]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk FilterMutectCalls \
  -R hg19_chr13_chr19.fa \
  -V WP017.somatic.vcf.gz \
  --contamination-table WP017.paired.contamination.table \
  -O WP017.filtered.vcf.gz

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar FilterMutectCalls -R hg19_chr13_chr19.fa -V WP017.somatic.vcf.gz --contamination-table WP017.paired.contamination.table -O WP017.filtered.vcf.gz
05:02:50.440 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
05:02:50.709 INFO  FilterMutectCalls - ------------------------------------------------------------
05:02:50.710 INFO  FilterMutectCalls - The Genome Analysis Toolkit (GATK) v4.2.2.0
05:02:50.711 INFO  FilterMutectCalls - For support and documentation go to https://software.broadinstitute.org/gatk/
05:02:50.711 INFO  FilterMutectCalls - Executin

In [140]:
import gzip
from collections import Counter

def ler_vcf(arquivo):
    variantes = {}
    with gzip.open(arquivo, "rt") as f:
        for linha in f:
            if linha.startswith("#"):
                continue
            c = linha.strip().split("\t")
            chave = (c[0], c[1], c[3], c[4])
            variantes[chave] = c[6]
    return variantes

paired = ler_vcf("/content/somatico/WP017.filtered.vcf.gz")
pon = ler_vcf("/content/somatico/WP017.filtered.pon.vcf.gz")

compartilhadas = set(paired) & set(pon)
exclusivas_paired = set(paired) - set(pon)
exclusivas_pon = set(pon) - set(paired)

print("=== RESUMO WP017 ===")
print("Variantes paired:", len(paired))
print("Variantes PoN:", len(pon))
print("Compartilhadas:", len(compartilhadas))
print("Exclusivas paired:", len(exclusivas_paired))
print("Exclusivas PoN:", len(exclusivas_pon))

print("\n=== COMPARTILHADAS ===")
for v in sorted(compartilhadas):
    print(v, "| paired:", paired[v], "| PoN:", pon[v])

print("\n=== EXCLUSIVAS PAIRED ===")
for v in sorted(exclusivas_paired):
    print(v, "|", paired[v])

print("\n=== EXCLUSIVAS PoN ===")
for v in sorted(exclusivas_pon):
    print(v, "|", pon[v])

=== RESUMO WP017 ===
Variantes paired: 3
Variantes PoN: 7
Compartilhadas: 3
Exclusivas paired: 0
Exclusivas PoN: 4

=== COMPARTILHADAS ===
('19', '13049389', 'T', 'G') | paired: base_qual;contamination;germline;strand_bias;weak_evidence | PoN: base_qual;contamination;germline;haplotype;weak_evidence
('19', '13049971', 'G', 'A') | paired: contamination;strand_bias;weak_evidence | PoN: contamination;weak_evidence
('19', '13054564', 'AGCAGAGGCTTAAGGAGGAGGAAGAAGACAAGAAACGCAAAGAGGAGGAGGAG', 'A') | paired: PASS | PoN: PASS

=== EXCLUSIVAS PAIRED ===

=== EXCLUSIVAS PoN ===
('19', '13049393', 'T', 'G') | base_qual;contamination;germline;haplotype
('19', '13053407', 'CTTT', 'C') | contamination;germline;weak_evidence
('19', '13053478', 'G', 'A') | contamination;germline;weak_evidence
('19', '13054781', 'G', 'T') | germline


In [141]:
%%bash
cd /content/somatico

echo "=== BAMs disponíveis ==="
find /content/somatico -type f \( \
  -iname "*wp043*.bam" -o \
  -iname "*wp044*.bam" -o \
  -iname "*043*.bam" -o \
  -iname "*044*.bam" \
\) -print

=== BAMs disponíveis ===


In [142]:
%%bash
find /content -type f -name "*.bam" -printf "%p\n" 2>/dev/null

/content/somatico/normal_JAK2 (1).bam
/content/somatico/tumor_wp191.bam
/content/somatico/tumor_wp017.bam
/content/somatico/normal_JAK2.bam
/content/somatico/tumor_JAK2.bam
/content/somatico/tumor_wp018.bam
/content/somatico/tumor_wp190.bam


In [143]:
%%bash
cd /content/somatico

for f in tumor_JAK2.bam normal_JAK2.bam "normal_JAK2 (1).bam"; do
    echo "===== $f ====="
    samtools view -H "$f" | grep '^@RG'
done

===== tumor_JAK2.bam =====
@RG	ID:1	PL:illumina	PU:WP043.sorted.ddup.bam	LB:tumor	SM:WP043
===== normal_JAK2.bam =====
@RG	ID:1	PL:illumina	PU:WP044.sorted.ddup.bam	LB:normal	SM:WP044
===== normal_JAK2 (1).bam =====
@RG	ID:1	PL:illumina	PU:WP044.sorted.ddup.bam	LB:normal	SM:WP044


In [144]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk Mutect2 \
  -R hg19_chr13_chr19.fa \
  -I tumor_JAK2.bam \
  -I normal_JAK2.bam \
  -normal WP044 \
  --germline-resource af-only-gnomad-chr13-chr19.vcf.gz \
  -O WP043.somatic.vcf.gz \
  -L hg19_chr13_chr19.interval_list

[0.002s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.002s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Tool returned:
SUCCESS


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar Mutect2 -R hg19_chr13_chr19.fa -I tumor_JAK2.bam -I normal_JAK2.bam -normal WP044 --germline-resource af-only-gnomad-chr13-chr19.vcf.gz -O WP043.somatic.vcf.gz -L hg19_chr13_chr19.interval_list
05:06:53.513 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
05:06:53.854 INFO  Mutect2 - ------------------------------------------------------------
05:06:53.855 INFO  Mutect2 - The Genome Analysis Toolkit (GATK) v4.2.2.0
05:06:53.855 INFO  Mutect2 - For support and documentation go to https://software.broadinstitute.org/gatk/
05:06:53.856 INFO  Mutect2 -

In [145]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk GetPileupSummaries \
  -I tumor_JAK2.bam \
  -V af-only-gnomad-chr13-chr19.vcf.gz \
  -L hg19_chr13_chr19.interval_list \
  -O WP043.paired.tumor.table

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Tool returned:
SUCCESS


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar GetPileupSummaries -I tumor_JAK2.bam -V af-only-gnomad-chr13-chr19.vcf.gz -L hg19_chr13_chr19.interval_list -O WP043.paired.tumor.table
05:08:17.712 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
05:08:18.169 INFO  GetPileupSummaries - ------------------------------------------------------------
05:08:18.171 INFO  GetPileupSummaries - The Genome Analysis Toolkit (GATK) v4.2.2.0
05:08:18.171 INFO  GetPileupSummaries - For support and documentation go to https://software.broadinstitute.org/gatk/
05:08:18.173 INFO  GetPileupSummaries - Executing as 

In [146]:
%%bash
cd /content/somatico

echo "=== TAMANHO DO BAM ==="
ls -lh tumor_JAK2.bam normal_JAK2.bam

echo "=== CONTIGS E LEITURAS DO TUMOR ==="
samtools idxstats tumor_JAK2.bam

echo "=== CONTIGS E LEITURAS DO NORMAL ==="
samtools idxstats normal_JAK2.bam

echo "=== CABEÇALHO DO TUMOR ==="
samtools view -H tumor_JAK2.bam | grep '^@SQ'

=== TAMANHO DO BAM ===
-rw-r--r-- 1 root root 574K Nov 25 02:10 normal_JAK2.bam
-rw-r--r-- 1 root root 997K Nov 25 02:34 tumor_JAK2.bam
=== CONTIGS E LEITURAS DO TUMOR ===
1	249250621	0	0
2	243199373	0	0
3	198022430	0	0
4	191154276	0	0
5	180915260	0	0
6	171115067	0	0
7	159138663	0	0
8	146364022	0	0
9	141213431	9056	0
10	135534747	0	0
11	135006516	0	0
12	133851895	0	0
13	115169878	0	0
14	107349540	0	0
15	102531392	0	0
16	90354753	0	0
17	81195210	0	0
18	78077248	0	0
19	59128983	0	0
20	63025520	0	0
21	48129895	0	0
22	51304566	0	0
X	155270560	0	0
Y	59373566	0	0
MT	16569	0	0
*	0	0	0
=== CONTIGS E LEITURAS DO NORMAL ===
1	249250621	0	0
2	243199373	0	0
3	198022430	0	0
4	191154276	0	0
5	180915260	0	0
6	171115067	0	0
7	159138663	0	0
8	146364022	0	0
9	141213431	4672	0
10	135534747	0	0
11	135006516	0	0
12	133851895	0	0
13	115169878	0	0
14	107349540	0	0
15	102531392	0	0
16	90354753	0	0
17	81195210	0	0
18	78077248	0	0
19	59128983	0	0
20	63025520	0	0
21	48129895	0	0
22	51304566	0	0
X	155270560	0	0
Y

In [147]:
%%bash
cd /content/somatico

echo "=== REFERÊNCIA ORIGINAL ==="
ls -lh hg19.fa*

echo "=== RECURSOS PoN ==="
ls -lh *panel* *pon* 2>/dev/null

echo "=== RECURSOS GNOMAD ==="
ls -lh *gnomad* 2>/dev/null

echo "=== INTERVALOS ==="
ls -lh *.interval_list

=== REFERÊNCIA ORIGINAL ===
-rw-r--r-- 1 root root 3.0G Nov 25 02:47 hg19.fa
-rw-r--r-- 1 root root 3.5K Nov 25 02:47 hg19.fa.fai
-rw-r--r-- 1 root root 905M Aug 21  2018 hg19.fa.gz
=== RECURSOS PoN ===
-rw-r--r-- 1 root root 9.6M Nov  6  2018 Mutect2-exome-panel.vcf
-rw-r--r-- 1 root root 8.6M Nov  6  2018 Mutect2-exome-panel.vcf.idx
-rw-r--r-- 1 root root  882 Nov 25 02:10 run.chr13-chr19.pon.sh
-rw-r--r-- 1 root root   73 Nov 25 04:55 WP017.contamination.pon.table
-rw-r--r-- 1 root root 5.4K Nov 25 04:56 WP017.filtered.pon.vcf.gz
-rw-r--r-- 1 root root 1.7K Nov 25 04:56 WP017.filtered.pon.vcf.gz.filteringStats.tsv
-rw-r--r-- 1 root root  129 Nov 25 04:56 WP017.filtered.pon.vcf.gz.tbi
-rw-r--r-- 1 root root  291 Nov 25 04:55 WP017.pon.table
-rw-r--r-- 1 root root 4.2K Nov 25 04:54 WP017.somatic.pon.vcf.gz
-rw-r--r-- 1 root root   32 Nov 25 04:54 WP017.somatic.pon.vcf.gz.stats
-rw-r--r-- 1 root root  129 Nov 25 04:54 WP017.somatic.pon.vcf.gz.tbi
-rw-r--r-- 1 root root   57 Nov 25 04:5

In [148]:
%%bash
cd /content/somatico

echo "=== REFERÊNCIA HG19 ==="
grep -E '^(9|chr9)[[:space:]]' hg19.fa.fai

echo "=== GNOMAD CHR9 ==="
grep '^#CHROM' af-only-gnomad-chr9.vcf
grep -v '^#' af-only-gnomad-chr9.vcf | head -3

echo "=== CONTIGS DO PoN ==="
grep '^##contig' Mutect2-exome-panel.vcf | head -15

echo "=== VARIANTES DO PoN NO CHR9 ==="
grep -v '^#' Mutect2-exome-panel.vcf | awk '$1=="9" || $1=="chr9"' | head -5

=== REFERÊNCIA HG19 ===
chr9	141213431	1728318943	50	51
=== GNOMAD CHR9 ===
9	5021940	.	A	G	1902.40	PASS	AC=2;AF=4.923E-5
9	5021993	.	A	C	506.46	PASS	AC=1;AF=2.462E-5
9	5022035	.	C	T	1311.40	PASS	AC=2;AF=4.923E-5
=== CONTIGS DO PoN ===
##contig=<ID=1,length=249250621>
##contig=<ID=2,length=243199373>
##contig=<ID=3,length=198022430>
##contig=<ID=4,length=191154276>
##contig=<ID=5,length=180915260>
##contig=<ID=6,length=171115067>
##contig=<ID=7,length=159138663>
##contig=<ID=8,length=146364022>
##contig=<ID=9,length=141213431>
##contig=<ID=10,length=135534747>
##contig=<ID=11,length=135006516>
##contig=<ID=12,length=133851895>
##contig=<ID=13,length=115169878>
##contig=<ID=14,length=107349540>
##contig=<ID=15,length=102531392>
=== VARIANTES DO PoN NO CHR9 ===
9	14764	.	C	T	.	PASS	SOMATIC
9	14765	.	G	A	.	PASS	SOMATIC
9	14788	.	G	A	.	PASS	SOMATIC
9	14810	.	C	G	.	PASS	SOMATIC
9	14811	.	G	A	.	PASS	SOMATIC


In [149]:
%%bash
cd /content/somatico

echo "=== INTERVALOS DO CROMOSSOMO 9 ==="
grep -E '^(9|chr9)[[:space:]]' hg19.interval_list | head -20

echo "=== CABEÇALHO DO INTERVAL LIST ==="
head -8 hg19.interval_list

=== INTERVALOS DO CROMOSSOMO 9 ===
chr9	10001	39663686	+	ACGTmer
chr9	39713687	39974796	+	ACGTmer
chr9	40024797	40233029	+	ACGTmer
chr9	40283030	40425834	+	ACGTmer
chr9	40475835	40940341	+	ACGTmer
chr9	40990342	41143214	+	ACGTmer
chr9	41193215	41365793	+	ACGTmer
chr9	41415794	42613955	+	ACGTmer
chr9	42663956	43213698	+	ACGTmer
chr9	43313699	43946569	+	ACGTmer
chr9	43996570	44676646	+	ACGTmer
chr9	44726647	44908293	+	ACGTmer
chr9	44958294	45250203	+	ACGTmer
chr9	45350204	45815521	+	ACGTmer
chr9	45865522	46216430	+	ACGTmer
chr9	46266431	46461039	+	ACGTmer
chr9	46561040	47060133	+	ACGTmer
chr9	47160134	47317679	+	ACGTmer
chr9	65467680	65918360	+	ACGTmer
chr9	65968361	66192215	+	ACGTmer
=== CABEÇALHO DO INTERVAL LIST ===
@HD	VN:1.6	SO:coordinate
@SQ	SN:chr1	LN:249250621	M5:1b22b98cdeb4a9304cb5d48026a85128	UR:file:/content/somatico/hg19.fa
@SQ	SN:chr2	LN:243199373	M5:a0d9851da00400dec1098a9255ac712e	UR:file:/content/somatico/hg19.fa
@SQ	SN:chr3	LN:198022430	M5:641e4338fa8d52a5b781bd2a2c08d3

In [150]:
%%bash
cd /content/somatico

echo "=== QUANTIDADE DE INTERVALOS CHR9 ==="
grep '^chr9' hg19.interval_list | wc -l

echo "=== PRIMEIROS INTERVALOS ==="
grep '^chr9' hg19.interval_list | head -5

echo "=== LEITURAS MAPEADAS NO CHR9 ==="
samtools view -c tumor_JAK2.bam 9

echo "=== LEITURAS DO TUMOR NAS REGIÕES EXÔMICAS ==="
samtools view -c -L <(grep '^chr9' hg19.interval_list | awk 'BEGIN{OFS="\t"} {print "9",$2,$3}') tumor_JAK2.bam

=== QUANTIDADE DE INTERVALOS CHR9 ===
43
=== PRIMEIROS INTERVALOS ===
chr9	10001	39663686	+	ACGTmer
chr9	39713687	39974796	+	ACGTmer
chr9	40024797	40233029	+	ACGTmer
chr9	40283030	40425834	+	ACGTmer
chr9	40475835	40940341	+	ACGTmer
=== LEITURAS MAPEADAS NO CHR9 ===
9056
=== LEITURAS DO TUMOR NAS REGIÕES EXÔMICAS ===
9056


In [151]:
%%bash
cd /content/somatico

python - <<'PY'
from pathlib import Path

origem = Path("hg19.fa")
destino = Path("hg19_chr9.fa")

copiando = False

with origem.open() as entrada, destino.open("w") as saida:
    for linha in entrada:
        if linha.startswith(">"):
            copiando = linha[1:].strip().split()[0] == "chr9"
            if copiando:
                saida.write(">9\n")
        elif copiando:
            saida.write(linha)

print("Referência chr9 criada:", destino)
PY

samtools faidx hg19_chr9.fa

./gatk-4.2.2.0/gatk CreateSequenceDictionary \
  -R hg19_chr9.fa \
  -O hg19_chr9.dict

echo "=== REFERÊNCIA CRIADA ==="
ls -lh hg19_chr9.fa*

Referência chr9 criada: hg19_chr9.fa
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Tool returned:
0
=== REFERÊNCIA CRIADA ===
-rw-r--r-- 1 root root 138M Nov 25 05:13 hg19_chr9.fa
-rw-r--r-- 1 root root   20 Nov 25 05:13 hg19_chr9.fa.fai


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar CreateSequenceDictionary -R hg19_chr9.fa -O hg19_chr9.dict
05:13:25.284 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
[Wed Nov 25 05:13:25 UTC 2026] CreateSequenceDictionary --OUTPUT hg19_chr9.dict --REFERENCE hg19_chr9.fa --TRUNCATE_NAMES_AT_WHITESPACE true --NUM_SEQUENCES 2147483647 --VERBOSITY INFO --QUIET false --VALIDATION_STRINGENCY STRICT --COMPRESSION_LEVEL 2 --MAX_RECORDS_IN_RAM 500000 --CREATE_INDEX false --CREATE_MD5_FILE false --GA4GH_CLIENT_SECRETS client_secrets.json --help false --version false --showHidden false --USE_JDK_DEFLATE

In [152]:
%%bash
cd /content/somatico

python - <<'PY'
from pathlib import Path

dict_file = Path("hg19_chr9.dict")
original = Path("hg19.interval_list")
output = Path("hg19_chr9.interval_list")

with dict_file.open() as f:
    sq_lines = [l for l in f if l.startswith("@SQ")]

with original.open() as f:
    hd_lines = [l for l in f if l.startswith("@HD")]
    intervals = [
        l.replace("chr9\t", "9\t", 1)
        for l in f
        if l.startswith("chr9\t")
    ]

with output.open("w") as f:
    f.writelines(hd_lines)
    f.writelines(sq_lines)
    f.writelines(intervals)

print("Intervalos criados:", len(intervals))
print("Arquivo:", output)
PY

echo "=== CABEÇALHO ==="
head -4 hg19_chr9.interval_list

echo "=== TOTAL DE INTERVALOS ==="
grep -v '^@' hg19_chr9.interval_list | wc -l

echo "=== CONTIG DO DICIONÁRIO ==="
grep '^@SQ' hg19_chr9.dict

Intervalos criados: 0
Arquivo: hg19_chr9.interval_list
=== CABEÇALHO ===
@HD	VN:1.6	SO:coordinate
@SQ	SN:9	LN:141213431	M5:3e273117f15e0a400f01055d9f393768	UR:file:/content/somatico/hg19_chr9.fa
=== TOTAL DE INTERVALOS ===
0
=== CONTIG DO DICIONÁRIO ===
@SQ	SN:9	LN:141213431	M5:3e273117f15e0a400f01055d9f393768	UR:file:/content/somatico/hg19_chr9.fa


In [153]:
%%bash
cd /content/somatico

echo "=== PRIMEIRAS LINHAS DOS INTERVALOS ORIGINAIS ==="
grep -v '^@' hg19.interval_list | head -5 | cat -A

echo "=== LINHAS INICIADAS POR chr9 ==="
grep '^chr9' hg19.interval_list | head -5 | cat -A

echo "=== TOTAL DE LINHAS chr9 ==="
grep -c '^chr9' hg19.interval_list

=== PRIMEIRAS LINHAS DOS INTERVALOS ORIGINAIS ===
chr1^I10001^I177417^I+^IACGTmer$
chr1^I227418^I267719^I+^IACGTmer$
chr1^I317720^I471368^I+^IACGTmer$
chr1^I521369^I2634220^I+^IACGTmer$
chr1^I2684221^I3845268^I+^IACGTmer$
=== LINHAS INICIADAS POR chr9 ===
chr9^I10001^I39663686^I+^IACGTmer$
chr9^I39713687^I39974796^I+^IACGTmer$
chr9^I40024797^I40233029^I+^IACGTmer$
chr9^I40283030^I40425834^I+^IACGTmer$
chr9^I40475835^I40940341^I+^IACGTmer$
=== TOTAL DE LINHAS chr9 ===
43


In [154]:
%%bash
cd /content/somatico

python - <<'PY'
from pathlib import Path

with open("hg19_chr9.dict") as f:
    sq = [l for l in f if l.startswith("@SQ")]

with open("hg19.interval_list") as f:
    hd = [l for l in f if l.startswith("@HD")]
    intervals = []
    for line in f:
        if line.startswith("@"):
            continue
        campos = line.rstrip("\n").split("\t")
        if campos[0] == "chr9":
            campos[0] = "9"
            intervals.append("\t".join(campos) + "\n")

with open("hg19_chr9.interval_list", "w") as f:
    f.writelines(hd)
    f.writelines(sq)
    f.writelines(intervals)

print("Intervalos criados:", len(intervals))
print("Primeiros intervalos:")
print("".join(intervals[:3]))
PY

echo "=== TOTAL FINAL ==="
grep -vc '^@' hg19_chr9.interval_list

Intervalos criados: 0
Primeiros intervalos:

=== TOTAL FINAL ===
0


CalledProcessError: Command 'b'cd /content/somatico\n\npython - <<\'PY\'\nfrom pathlib import Path\n\nwith open("hg19_chr9.dict") as f:\n    sq = [l for l in f if l.startswith("@SQ")]\n\nwith open("hg19.interval_list") as f:\n    hd = [l for l in f if l.startswith("@HD")]\n    intervals = []\n    for line in f:\n        if line.startswith("@"):\n            continue\n        campos = line.rstrip("\\n").split("\\t")\n        if campos[0] == "chr9":\n            campos[0] = "9"\n            intervals.append("\\t".join(campos) + "\\n")\n\nwith open("hg19_chr9.interval_list", "w") as f:\n    f.writelines(hd)\n    f.writelines(sq)\n    f.writelines(intervals)\n\nprint("Intervalos criados:", len(intervals))\nprint("Primeiros intervalos:")\nprint("".join(intervals[:3]))\nPY\n\necho "=== TOTAL FINAL ==="\ngrep -vc \'^@\' hg19_chr9.interval_list\n'' returned non-zero exit status 1.

In [155]:
%%bash
cd /content/somatico

python - <<'PY'
from pathlib import Path

linhas = Path("hg19.interval_list").read_text().splitlines()

cabecalho_hd = [x + "\n" for x in linhas if x.startswith("@HD")]
intervalos = []

for linha in linhas:
    if linha.startswith("@"):
        continue

    campos = linha.split()

    if campos and campos[0] == "chr9":
        campos[0] = "9"
        intervalos.append("\t".join(campos) + "\n")

sq = [
    x + "\n"
    for x in Path("hg19_chr9.dict").read_text().splitlines()
    if x.startswith("@SQ")
]

Path("hg19_chr9.interval_list").write_text(
    "".join(cabecalho_hd + sq + intervalos)
)

print("Intervalos criados:", len(intervalos))
print("Primeiro intervalo:", intervalos[0].strip() if intervalos else "NENHUM")
print("Último intervalo:", intervalos[-1].strip() if intervalos else "NENHUM")
PY

Intervalos criados: 39
Primeiro intervalo: 9	10001	39663686	+	ACGTmer
Último intervalo: 9	139216998	141153431	+	ACGTmer


In [156]:
%%bash
cd /content/somatico

echo "=== CONTAGEM POR FORMATO DE LINHA ==="
grep '^chr9' hg19.interval_list | awk '{print NF}' | sort | uniq -c

echo "=== LINHAS chr9 QUE NÃO TÊM 5 COLUNAS ==="
grep '^chr9' hg19.interval_list | awk 'NF != 5 {print NR, NF, $0}'

echo "=== CONTAGEM TOTAL ==="
grep -c '^chr9' hg19.interval_list

=== CONTAGEM POR FORMATO DE LINHA ===
     43 5
=== LINHAS chr9 QUE NÃO TÊM 5 COLUNAS ===
=== CONTAGEM TOTAL ===
43


In [157]:
%%bash
cd /content/somatico

{
  grep '^@HD' hg19.interval_list
  grep '^@SQ' hg19_chr9.dict
  awk '$1=="chr9" {$1="9"; OFS="\t"; print}' hg19.interval_list
} > hg19_chr9.interval_list

echo "=== INTERVALOS CRIADOS ==="
grep -vc '^@' hg19_chr9.interval_list

echo "=== PRIMEIRAS LINHAS ==="
head -5 hg19_chr9.interval_list

echo "=== ÚLTIMAS LINHAS ==="
tail -3 hg19_chr9.interval_list

=== INTERVALOS CRIADOS ===
39
=== PRIMEIRAS LINHAS ===
@HD	VN:1.6	SO:coordinate
@SQ	SN:9	LN:141213431	M5:3e273117f15e0a400f01055d9f393768	UR:file:/content/somatico/hg19_chr9.fa
9 10001 39663686 + ACGTmer
9	39713687	39974796	+	ACGTmer
9	40024797	40233029	+	ACGTmer
=== ÚLTIMAS LINHAS ===
9	133223061	137041193	+	ACGTmer
9	137091194	139166997	+	ACGTmer
9	139216998	141153431	+	ACGTmer


In [158]:
%%bash
cd /content/somatico

{
  grep '^@HD' hg19.interval_list
  grep '^@SQ' hg19_chr9.dict
  grep '^chr9' hg19.interval_list | sed 's/^chr9/9/'
} > hg19_chr9.interval_list

echo "=== TOTAL DE INTERVALOS ==="
grep -vc '^@' hg19_chr9.interval_list

echo "=== CONTAGEM DE CONTIG 9 ==="
grep -c '^9[[:space:]]' hg19_chr9.interval_list

echo "=== PRIMEIRO INTERVALO ==="
grep -v '^@' hg19_chr9.interval_list | head -1

echo "=== ÚLTIMO INTERVALO ==="
grep -v '^@' hg19_chr9.interval_list | tail -1

=== TOTAL DE INTERVALOS ===
43
=== CONTAGEM DE CONTIG 9 ===
39
=== PRIMEIRO INTERVALO ===
9	10001	39663686	+	ACGTmer
=== ÚLTIMO INTERVALO ===
9_gl000201_random	1	36148	+	ACGTmer


In [159]:
%%bash
cd /content/somatico

{
  grep '^@HD' hg19.interval_list
  grep '^@SQ' hg19_chr9.dict
  grep '^chr9[[:space:]]' hg19.interval_list | sed 's/^chr9/9/'
} > hg19_chr9.interval_list

echo "=== TOTAL DE INTERVALOS ==="
grep -vc '^@' hg19_chr9.interval_list

echo "=== CONTIGS ==="
grep -v '^@' hg19_chr9.interval_list | awk '{print $1}' | sort | uniq -c

echo "=== CABEÇALHO ==="
head -3 hg19_chr9.interval_list

=== TOTAL DE INTERVALOS ===
39
=== CONTIGS ===
     39 9
=== CABEÇALHO ===
@HD	VN:1.6	SO:coordinate
@SQ	SN:9	LN:141213431	M5:3e273117f15e0a400f01055d9f393768	UR:file:/content/somatico/hg19_chr9.fa
9	10001	39663686	+	ACGTmer


In [160]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk SelectVariants \
  -R hg19_chr9.fa \
  -V Mutect2-exome-panel.vcf \
  -L hg19_chr9.interval_list \
  -O Mutect2-exome-panel-chr9.vcf

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar SelectVariants -R hg19_chr9.fa -V Mutect2-exome-panel.vcf -L hg19_chr9.interval_list -O Mutect2-exome-panel-chr9.vcf
05:19:22.637 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
05:19:22.934 INFO  SelectVariants - ------------------------------------------------------------
05:19:22.935 INFO  SelectVariants - The Genome Analysis Toolkit (GATK) v4.2.2.0
05:19:22.935 INFO  SelectVariants - For support and documentation go to https://software.broadinstitute.org/gatk/
05:19:22.936 INFO  SelectVariants - Executing as root@97750e0669bd on Linux v6.6.159

In [161]:
%%bash
cd /content/somatico

echo "=== VARIANTES NO PoN chr9 ==="
grep -vc '^#' Mutect2-exome-panel-chr9.vcf

echo "=== CABEÇALHO E CONTIGS ==="
grep '^##contig' Mutect2-exome-panel-chr9.vcf | head

echo "=== CRIAR ÍNDICE ==="
./gatk-4.2.2.0/gatk IndexFeatureFile \
  -I Mutect2-exome-panel-chr9.vcf

=== VARIANTES NO PoN chr9 ===
12599
=== CABEÇALHO E CONTIGS ===
##contig=<ID=9,length=141213431,assembly=hg19_chr9.fa>
=== CRIAR ÍNDICE ===
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Tool returned:
/content/somatico/Mutect2-exome-panel-chr9.vcf.idx


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar IndexFeatureFile -I Mutect2-exome-panel-chr9.vcf
05:19:50.288 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
05:19:50.552 INFO  IndexFeatureFile - ------------------------------------------------------------
05:19:50.552 INFO  IndexFeatureFile - The Genome Analysis Toolkit (GATK) v4.2.2.0
05:19:50.553 INFO  IndexFeatureFile - For support and documentation go to https://software.broadinstitute.org/gatk/
05:19:50.553 INFO  IndexFeatureFile - Executing as root@97750e0669bd on Linux v6.6.159+ amd64
05:19:50.553 INFO  IndexFeatureFile - Java runtime: 

In [162]:
%%bash
cd /content/somatico

echo "=== ARQUIVOS GNOMAD CHR9 ==="
ls -lh af-only-gnomad-chr9.vcf.gz*

echo "=== CONTIGS DO GNOMAD ==="
zgrep '^##contig' af-only-gnomad-chr9.vcf.gz | head

echo "=== PRIMEIRAS VARIANTES ==="
zgrep -v '^#' af-only-gnomad-chr9.vcf.gz | head -3

echo "=== REFERÊNCIA ==="
cat hg19_chr9.fa.fai

=== ARQUIVOS GNOMAD CHR9 ===
-rw-r--r-- 1 root root 138K Nov 25 02:10 af-only-gnomad-chr9.vcf.gz
-rw-r--r-- 1 root root  194 Nov 25 02:42 af-only-gnomad-chr9.vcf.gz.tbi
=== CONTIGS DO GNOMAD ===
=== PRIMEIRAS VARIANTES ===
9	5021940	.	A	G	1902.40	PASS	AC=2;AF=4.923E-5
9	5021993	.	A	C	506.46	PASS	AC=1;AF=2.462E-5
9	5022035	.	C	T	1311.40	PASS	AC=2;AF=4.923E-5
=== REFERÊNCIA ===
9	141213431	3	50	51


In [163]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk GetPileupSummaries \
  -I tumor_JAK2.bam \
  -V af-only-gnomad-chr9.vcf.gz \
  -L hg19_chr9.interval_list \
  -O WP043.paired.tumor.table

./gatk-4.2.2.0/gatk GetPileupSummaries \
  -I normal_JAK2.bam \
  -V af-only-gnomad-chr9.vcf.gz \
  -L hg19_chr9.interval_list \
  -O WP043.paired.normal.table

echo "=== TUMOR: LINHAS DA TABELA ==="
wc -l WP043.paired.tumor.table

echo "=== NORMAL: LINHAS DA TABELA ==="
wc -l WP043.paired.normal.table

echo "=== CONTEÚDO TUMOR ==="
head -5 WP043.paired.tumor.table

echo "=== CONTEÚDO NORMAL ==="
head -5 WP043.paired.normal.table

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Tool returned:
SUCCESS
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Tool returned:
SUCCESS
=== TUMOR: LINHAS DA TABELA ===
64 WP043.paired.tumor.table
=== NORMAL: LINHAS DA TABELA ===
39 WP043.paired.normal.table
=== CONTEÚDO TUMOR ===
#<METADATA>SAMPLE=WP043
contig	position	ref_count	alt_count	other_alt_count	allele_frequency
9	5029611	6	0	0	0.012
9	5030080	3	0	0	0.078
9	5033977	1	0	

Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar GetPileupSummaries -I tumor_JAK2.bam -V af-only-gnomad-chr9.vcf.gz -L hg19_chr9.interval_list -O WP043.paired.tumor.table
05:21:05.664 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
05:21:05.959 INFO  GetPileupSummaries - ------------------------------------------------------------
05:21:05.961 INFO  GetPileupSummaries - The Genome Analysis Toolkit (GATK) v4.2.2.0
05:21:05.961 INFO  GetPileupSummaries - For support and documentation go to https://software.broadinstitute.org/gatk/
05:21:05.961 INFO  GetPileupSummaries - Executing as root@97750e066

In [164]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk CalculateContamination \
  -I WP043.paired.tumor.table \
  -matched WP043.paired.normal.table \
  -O WP043.paired.contamination.table \
  --tumor-segmentation WP043.paired.segments.table

echo "=== CONTAMINAÇÃO WP043 ==="
cat WP043.paired.contamination.table

echo "=== SEGMENTAÇÃO ==="
head WP043.paired.segments.table

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Tool returned:
SUCCESS
=== CONTAMINAÇÃO WP043 ===
sample	contamination	error
WP043	0.0	4.76837158203125E-7
=== SEGMENTAÇÃO ===
#<METADATA>SAMPLE=WP043
contig	start	end	minor_allele_fraction


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar CalculateContamination -I WP043.paired.tumor.table -matched WP043.paired.normal.table -O WP043.paired.contamination.table --tumor-segmentation WP043.paired.segments.table
05:21:56.739 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
05:21:57.008 INFO  CalculateContamination - ------------------------------------------------------------
05:21:57.009 INFO  CalculateContamination - The Genome Analysis Toolkit (GATK) v4.2.2.0
05:21:57.009 INFO  CalculateContamination - For support and documentation go to https://software.broadinstitute.org/gatk/
05:21:

In [165]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk Mutect2 \
  -R hg19_chr9.fa \
  -I tumor_JAK2.bam \
  -I normal_JAK2.bam \
  -normal WP044 \
  --germline-resource af-only-gnomad-chr9.vcf.gz \
  -O WP043.somatic.chr9.vcf.gz \
  -L hg19_chr9.interval_list

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Tool returned:
SUCCESS


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar Mutect2 -R hg19_chr9.fa -I tumor_JAK2.bam -I normal_JAK2.bam -normal WP044 --germline-resource af-only-gnomad-chr9.vcf.gz -O WP043.somatic.chr9.vcf.gz -L hg19_chr9.interval_list
05:22:29.961 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
05:22:30.430 INFO  Mutect2 - ------------------------------------------------------------
05:22:30.431 INFO  Mutect2 - The Genome Analysis Toolkit (GATK) v4.2.2.0
05:22:30.432 INFO  Mutect2 - For support and documentation go to https://software.broadinstitute.org/gatk/
05:22:30.432 INFO  Mutect2 - Executing as ro

In [166]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk FilterMutectCalls \
  -R hg19_chr9.fa \
  -V WP043.somatic.chr9.vcf.gz \
  --contamination-table WP043.paired.contamination.table \
  -O WP043.filtered.chr9.vcf.gz

echo "=== TOTAL DE VARIANTES FILTRADAS ==="
zgrep -vc '^#' WP043.filtered.chr9.vcf.gz

echo "=== CLASSIFICAÇÃO DOS FILTROS ==="
zgrep -v '^#' WP043.filtered.chr9.vcf.gz | cut -f7 | sort | uniq -c

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
=== TOTAL DE VARIANTES FILTRADAS ===
4
=== CLASSIFICAÇÃO DOS FILTROS ===
      1 normal_artifact;slippage;strand_bias;weak_evidence
      1 normal_artifact;strand_bias;weak_evidence
      1 PASS
      1 strand_bias


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar FilterMutectCalls -R hg19_chr9.fa -V WP043.somatic.chr9.vcf.gz --contamination-table WP043.paired.contamination.table -O WP043.filtered.chr9.vcf.gz
05:23:57.688 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
05:23:57.936 INFO  FilterMutectCalls - ------------------------------------------------------------
05:23:57.937 INFO  FilterMutectCalls - The Genome Analysis Toolkit (GATK) v4.2.2.0
05:23:57.937 INFO  FilterMutectCalls - For support and documentation go to https://software.broadinstitute.org/gatk/
05:23:57.938 INFO  FilterMutectCalls - Execu

In [167]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk Mutect2 \
  -R hg19_chr9.fa \
  -I tumor_JAK2.bam \
  --germline-resource af-only-gnomad-chr9.vcf.gz \
  --panel-of-normals Mutect2-exome-panel-chr9.vcf \
  -O WP043.pon.chr9.vcf.gz \
  -L hg19_chr9.interval_list

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Tool returned:
SUCCESS


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar Mutect2 -R hg19_chr9.fa -I tumor_JAK2.bam --germline-resource af-only-gnomad-chr9.vcf.gz --panel-of-normals Mutect2-exome-panel-chr9.vcf -O WP043.pon.chr9.vcf.gz -L hg19_chr9.interval_list
05:24:53.115 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
05:24:53.683 INFO  Mutect2 - ------------------------------------------------------------
05:24:53.686 INFO  Mutect2 - The Genome Analysis Toolkit (GATK) v4.2.2.0
05:24:53.686 INFO  Mutect2 - For support and documentation go to https://software.broadinstitute.org/gatk/
05:24:53.688 INFO  Mutect2 - Exec

In [168]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk GetPileupSummaries \
  -I tumor_JAK2.bam \
  -V af-only-gnomad-chr9.vcf.gz \
  -L hg19_chr9.interval_list \
  -O WP043.pon.tumor.table

./gatk-4.2.2.0/gatk CalculateContamination \
  -I WP043.pon.tumor.table \
  -O WP043.pon.contamination.table

echo "=== CONTAMINAÇÃO TUMOR-ONLY ==="
cat WP043.pon.contamination.table

[0.012s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.012s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Tool returned:
SUCCESS
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Tool returned:
SUCCESS
=== CONTAMINAÇÃO TUMOR-ONLY ===
sample	contamination	error
WP043	0.0	4.76837158203125E-7


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar GetPileupSummaries -I tumor_JAK2.bam -V af-only-gnomad-chr9.vcf.gz -L hg19_chr9.interval_list -O WP043.pon.tumor.table
05:26:39.529 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
05:26:40.167 INFO  GetPileupSummaries - ------------------------------------------------------------
05:26:40.168 INFO  GetPileupSummaries - The Genome Analysis Toolkit (GATK) v4.2.2.0
05:26:40.168 INFO  GetPileupSummaries - For support and documentation go to https://software.broadinstitute.org/gatk/
05:26:40.173 INFO  GetPileupSummaries - Executing as root@97750e0669bd

In [169]:
%%bash
cd /content/somatico

./gatk-4.2.2.0/gatk FilterMutectCalls \
  -R hg19_chr9.fa \
  -V WP043.pon.chr9.vcf.gz \
  --contamination-table WP043.pon.contamination.table \
  -O WP043.filtered.pon.chr9.vcf.gz

echo "=== TOTAL DE VARIANTES FILTRADAS — PoN ==="
zgrep -vc '^#' WP043.filtered.pon.chr9.vcf.gz

echo "=== CLASSIFICAÇÃO DOS FILTROS — PoN ==="
zgrep -v '^#' WP043.filtered.pon.chr9.vcf.gz | cut -f7 | sort | uniq -c

[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
=== TOTAL DE VARIANTES FILTRADAS — PoN ===
22
=== CLASSIFICAÇÃO DOS FILTROS — PoN ===
      3 clustered_events;haplotype;weak_evidence
      1 germline;slippage;weak_evidence
      1 germline;weak_evidence
     14 PASS
      1 slippage;weak_evidence
      2 weak_evidence


Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar FilterMutectCalls -R hg19_chr9.fa -V WP043.pon.chr9.vcf.gz --contamination-table WP043.pon.contamination.table -O WP043.filtered.pon.chr9.vcf.gz
05:27:35.011 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
05:27:35.491 INFO  FilterMutectCalls - ------------------------------------------------------------
05:27:35.492 INFO  FilterMutectCalls - The Genome Analysis Toolkit (GATK) v4.2.2.0
05:27:35.492 INFO  FilterMutectCalls - For support and documentation go to https://software.broadinstitute.org/gatk/
05:27:35.493 INFO  FilterMutectCalls - Executin

In [170]:
import gzip

def ler_vcf(arquivo):
    variantes = {}
    with gzip.open(arquivo, "rt") as f:
        for linha in f:
            if linha.startswith("#"):
                continue
            campos = linha.strip().split("\t")
            chave = (campos[0], campos[1], campos[3], campos[4])
            variantes[chave] = campos[6]
    return variantes

paired = ler_vcf("WP043.filtered.chr9.vcf.gz")
pon = ler_vcf("WP043.filtered.pon.chr9.vcf.gz")

compartilhadas = sorted(set(paired) & set(pon))
exclusivas_paired = sorted(set(paired) - set(pon))
exclusivas_pon = sorted(set(pon) - set(paired))

print("=== RESUMO WP043 ===")
print("Variantes paired:", len(paired))
print("Variantes PoN:", len(pon))
print("Compartilhadas:", len(compartilhadas))
print("Exclusivas paired:", len(exclusivas_paired))
print("Exclusivas PoN:", len(exclusivas_pon))

print("\n=== VARIANTES COMPARTILHADAS ===")
for v in compartilhadas:
    print(
        f"{v[0]}:{v[1]} {v[2]}>{v[3]} | "
        f"paired: {paired[v]} | PoN: {pon[v]}"
    )

print("\n=== EXCLUSIVAS PAIRED ===")
for v in exclusivas_paired:
    print(f"{v[0]}:{v[1]} {v[2]}>{v[3]} | paired: {paired[v]}")

print("\n=== EXCLUSIVAS PoN ===")
for v in exclusivas_pon:
    print(f"{v[0]}:{v[1]} {v[2]}>{v[3]} | PoN: {pon[v]}")

=== RESUMO WP043 ===
Variantes paired: 4
Variantes PoN: 22
Compartilhadas: 4
Exclusivas paired: 0
Exclusivas PoN: 18

=== VARIANTES COMPARTILHADAS ===
9:5044364 C>A | paired: normal_artifact;strand_bias;weak_evidence | PoN: PASS
9:5054761 A>G | paired: strand_bias | PoN: PASS
9:5073681 C>CT | paired: normal_artifact;slippage;strand_bias;weak_evidence | PoN: slippage;weak_evidence
9:5073770 G>T | paired: PASS | PoN: PASS

=== EXCLUSIVAS PAIRED ===

=== EXCLUSIVAS PoN ===
9:5026771 T>A | PoN: PASS
9:5032460 C>A | PoN: weak_evidence
9:5032715 G>T | PoN: PASS
9:5033052 G>A | PoN: PASS
9:5035558 G>A | PoN: clustered_events;haplotype;weak_evidence
9:5035568 G>C | PoN: clustered_events;haplotype;weak_evidence
9:5035578 C>T | PoN: clustered_events;haplotype;weak_evidence
9:5064256 G>T | PoN: PASS
9:5067804 C>A | PoN: PASS
9:5076360 T>C | PoN: PASS
9:5082218 C>A | PoN: weak_evidence
9:5101126 G>T | PoN: PASS
9:5101271 G>T | PoN: PASS
9:5111548 T>TG | PoN: germline;weak_evidence
9:5113796 C>A | 

In [172]:

def localizar_vcf(amostra, modalidade):

    if modalidade == "paired":
        candidatos = [
            PASTA / f"{amostra}.somatic.vcf.gz",
            PASTA / f"{amostra}.somatic.chr9.vcf.gz",
            PASTA / f"{amostra}.filtered.chr9.vcf.gz"
        ]

    elif modalidade == "pon":
        candidatos = [
            PASTA / f"{amostra}.filtered.pon.chr9.vcf.gz",
            PASTA / f"{amostra}.pon.chr9.vcf.gz"
        ]

    for arquivo in candidatos:
        if arquivo.exists():
            print(f"{amostra} | {modalidade}: {arquivo.name}")
            return arquivo

    raise FileNotFoundError(
        f"VCF não encontrado para {amostra} ({modalidade})."
    )


In [174]:
import gzip
import pandas as pd
import os

# ==========================================================
# 1. ARQUIVOS DE ENTRADA
# ==========================================================

amostras = {
    "WP043": {
        "paired": "WP043.filtered.chr9.vcf.gz",
        "pon": "WP043.filtered.pon.chr9.vcf.gz"
    },
    "WP017": {
        "paired": "WP017.filtered.vcf.gz",
        "pon": "WP017.filtered.pon.vcf.gz"
    },
    "WP190": {
        "paired": "WP190.filtered.vcf.gz",
        "pon": "WP190.filtered.pon.vcf.gz"
    }
}

# ==========================================================
# 2. FUNÇÃO PARA LER OS VCFs
# ==========================================================

def ler_vcf(arquivo):
    variantes = {}

    if not os.path.exists(arquivo):
        raise FileNotFoundError(f"Arquivo não encontrado: {arquivo}")

    with gzip.open(arquivo, "rt") as f:
        for linha in f:
            if linha.startswith("#"):
                continue

            campos = linha.strip().split("\t")

            cromossomo = campos[0]
            posicao = campos[1]
            ref = campos[3]
            alt = campos[4]
            filtro = campos[6]

            chave = (cromossomo, posicao, ref, alt)

            variantes[chave] = filtro

    return variantes

# ==========================================================
# 3. COMPARAÇÃO ENTRE PAIRED E PoN
# ==========================================================

resultados = []
resumo = []

for amostra, arquivos in amostras.items():

    paired = ler_vcf(arquivos["paired"])
    pon = ler_vcf(arquivos["pon"])

    compartilhadas = set(paired) & set(pon)
    exclusivas_paired = set(paired) - set(pon)
    exclusivas_pon = set(pon) - set(paired)

    resumo.append({
        "Amostra": amostra,
        "Variantes_Paired": len(paired),
        "Variantes_PoN": len(pon),
        "Compartilhadas": len(compartilhadas),
        "Exclusivas_Paired": len(exclusivas_paired),
        "Exclusivas_PoN": len(exclusivas_pon),
        "Recuperacao_PoN_%": round(
            len(compartilhadas) / len(paired) * 100, 2
        ) if paired else 0
    })

    todas = sorted(set(paired) | set(pon))

    for variante in todas:

        cromossomo, posicao, ref, alt = variante

        filtro_paired = paired.get(variante, "AUSENTE")
        filtro_pon = pon.get(variante, "AUSENTE")

        if variante in compartilhadas:
            status = "COMPARTILHADA"
        elif variante in exclusivas_paired:
            status = "EXCLUSIVA_PAIRED"
        else:
            status = "EXCLUSIVA_PoN"

        resultados.append({
            "Amostra": amostra,
            "Cromossomo": cromossomo,
            "Posicao": posicao,
            "REF": ref,
            "ALT": alt,
            "Variante": f"{cromossomo}:{posicao} {ref}>{alt}",
            "Filtro_Paired": filtro_paired,
            "Filtro_PoN": filtro_pon,
            "Status": status
        })

# ==========================================================
# 4. CRIAÇÃO DAS TABELAS
# ==========================================================

df_variantes = pd.DataFrame(resultados)

df_resumo = pd.DataFrame(resumo)

# Ordenação
df_variantes = df_variantes.sort_values(
    by=["Amostra", "Cromossomo", "Posicao"]
)

# ==========================================================
# 5. RESUMO GERAL
# ==========================================================

print("\n========== RESUMO POR AMOSTRA ==========\n")
print(df_resumo.to_string(index=False))

print("\n========== RESUMO GERAL ==========\n")

print("Total de variantes únicas:", len(df_variantes))

print("\nClassificação geral:")
print(df_variantes["Status"].value_counts())

print("\nVariantes PASS no Paired:")
print((df_variantes["Filtro_Paired"] == "PASS").sum())

print("\nVariantes PASS no PoN:")
print((df_variantes["Filtro_PoN"] == "PASS").sum())

print("\nVariantes compartilhadas com filtros diferentes:")
comp = df_variantes[df_variantes["Status"] == "COMPARTILHADA"]

diferentes = comp[
    comp["Filtro_Paired"] != comp["Filtro_PoN"]
]

print(len(diferentes))

# ==========================================================
# 6. EXPORTAÇÃO
# ==========================================================

arquivo_excel = "Comparacao_Final_Paired_vs_PoN.xlsx"
arquivo_csv = "Comparacao_Final_Paired_vs_PoN.csv"

with pd.ExcelWriter(arquivo_excel, engine="openpyxl") as writer:

    df_resumo.to_excel(
        writer,
        sheet_name="Resumo",
        index=False
    )

    df_variantes.to_excel(
        writer,
        sheet_name="Todas_as_variantes",
        index=False
    )

    diferentes.to_excel(
        writer,
        sheet_name="Filtros_diferentes",
        index=False
    )

df_variantes.to_csv(
    arquivo_csv,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print("\n========== ARQUIVOS GERADOS ==========")
print(arquivo_excel)
print(arquivo_csv)


========== RESUMO POR AMOSTRA ==========

Amostra  Variantes_Paired  Variantes_PoN  Compartilhadas  Exclusivas_Paired  Exclusivas_PoN  Recuperacao_PoN_%
  WP043                 4             22               4                  0              18              100.0
  WP017                 3              7               3                  0               4              100.0
  WP190                 5             35               5                  0              30              100.0

========== RESUMO GERAL ==========

Total de variantes únicas: 64

Classificação geral:
Status
EXCLUSIVA_PoN    52
COMPARTILHADA    12
Name: count, dtype: int64

Variantes PASS no Paired:
4

Variantes PASS no PoN:
18

Variantes compartilhadas com filtros diferentes:
5

========== ARQUIVOS GERADOS ==========
Comparacao_Final_Paired_vs_PoN.xlsx
Comparacao_Final_Paired_vs_PoN.csv
